In [3]:
import os
import re
import html

import pandas as pd
import requests
from dotenv import load_dotenv

# 프로젝트의 .env 파일에서 API 키 불러오기
load_dotenv("/Users/anna/bulsae/.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

# 키가 있는지만 확인하고, 실제 값은 출력하지 않음
if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에서 네이버 API 키를 찾을 수 없습니다.")

print("네이버 API 키를 확인했습니다.")

네이버 API 키를 확인했습니다.


In [4]:
# NAVER API HUB의 뉴스 검색 주소
url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

# API 인증 정보
headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

# 검색 조건
params = {
    "query": "아모레퍼시픽",
    "display": 10,  # 결과 10건
    "start": 1,     # 첫 번째 결과부터
    "sort": "date", # 최신순
}

# 검색 요청
response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30,
)

# 요청에 문제가 있으면 오류 표시
response.raise_for_status()

# API 응답을 파이썬 데이터로 변환
news_data = response.json()

print("검색 결과 수:", news_data.get("total"))
print("이번에 가져온 기사 수:", len(news_data.get("items", [])))

검색 결과 수: 376237
이번에 가져온 기사 수: 10


In [5]:
# 네이버 검색 결과의 HTML 태그와 특수문자 정리
def clean_news_text(text):
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 검색 결과를 담을 리스트
news_rows = []

for item in news_data.get("items", []):
    news_rows.append({
        "title": clean_news_text(item.get("title")),
        "description": clean_news_text(item.get("description")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "published_at": item.get("pubDate"),
    })

# 리스트를 표로 변환
news_df = pd.DataFrame(news_rows)

print(f"기사 후보: {len(news_df)}건")
display(news_df)

기사 후보: 10건


,title,description,original_url,naver_url,published_at
0,국회 입법박람회 개막...로봇·수소차·누리호 등 첨단기술 한자리에,"한화오션은 친환경 액화천연가스(LNG) 운반선 모형을 전시하고, 아모레퍼시픽은 AI...",https://www.m-economynews.com/news/article.htm...,https://www.m-economynews.com/news/article.htm...,"Thu, 05 Nov 2026 18:04:00 +0900"
1,[기획] K뷰티 수출 신기록에도 화장품주 약세,"코스맥스와 한국콜마 역시 각각 3.67%, 2.39% 하락한 반면 아모레퍼시픽은 3...",https://www.kfenews.co.kr/news/articleView.htm...,https://www.kfenews.co.kr/news/articleView.htm...,"Thu, 05 Nov 2026 17:58:00 +0900"
2,'보고 체험하고 제안하는' 2026 국회 입법박람회 개막,"이 밖에도 한국과학기술원(KAIST)의 휴머노이드·4족 보행 로봇 시연, 한화오션의...",https://www.wikileaks-kr.org/news/articleView....,https://www.wikileaks-kr.org/news/articleView....,"Thu, 05 Nov 2026 17:50:00 +0900"
3,"패션주 영원 계열 큰폭 상승, 뷰티주 강세 두드러져","이밖에 아모레퍼시픽홀딩스(+4.37%), 한국화장품제조(+4.36%), 아모레퍼시픽...",http://okfashion.co.kr/detail.php?number=64346...,http://okfashion.co.kr/detail.php?number=64346...,"Thu, 05 Nov 2026 17:46:00 +0900"
4,K뷰티 역대급 호황에도 실적 ‘희비’,하지만 화장품 기업들의 3분기 성적표는 엇갈렸습니다.아모레퍼시픽은 더마뷰티 브랜드의...,https://news.mtn.co.kr/article/202611051551122...,https://news.mtn.co.kr/article/202611051551122...,"Thu, 05 Nov 2026 17:46:00 +0900"
5,"에이피 뷰티, '크림을 미스트로'… M.D. 라인 신제품 출시",크림 미스트/ 아모레퍼시픽 제공 아모레퍼시픽의 하이엔드 스킨케어 브랜드 에이피 뷰티...,https://www.koreaittimes.com/news/articleView....,https://www.koreaittimes.com/news/articleView....,"Thu, 05 Nov 2026 17:20:00 +0900"
6,"日 Z세대 ""K-뷰티, 매력적""…큐텐 재팬 '메가포' 상위권 장악","랭킹에는 아이보들, 셀리맥스, 블랑네이처, 아누아 등 인디 브랜드부터 에이피알 메디...",http://www.hansbiz.co.kr/news/articleView.html...,http://www.hansbiz.co.kr/news/articleView.html...,"Thu, 05 Nov 2026 17:20:00 +0900"
7,"신약 연구 기술, 화장품으로…바이오업계, 제품화·판로 확대 모색","박원석 아모레퍼시픽 R&I 선행연구부 상무는 ""흥미로운 기술과 제품이 되는 기술은 ...",https://www.financialpost.co.kr/news/articleVi...,https://www.financialpost.co.kr/news/articleVi...,"Thu, 05 Nov 2026 17:20:00 +0900"
8,"아모레퍼시픽 에이피 뷰티, 피부 보습·탄력 관리 '크림 미스트' 출시",현대경제신문 전수연 기자 | 아모레퍼시픽의 하이엔드 스킨케어 브랜드 에이피 뷰티(A...,http://www.finomy.com/news/articleView.html?id...,http://www.finomy.com/news/articleView.html?id...,"Thu, 05 Nov 2026 17:00:00 +0900"
9,"아모레퍼시픽 중국 C레벨 교체 이후 반등 신호, 김승환 설화수 흥행 기...",김승환 아모레퍼시픽 대표이사 사장이 중국 사업에서 매출 반등의 실마리를 찾고 있다....,https://www.businesspost.co.kr/BP?command=arti...,https://www.businesspost.co.kr/BP?command=arti...,"Thu, 05 Nov 2026 16:58:00 +0900"


In [6]:
from bs4 import BeautifulSoup
from markdownify import markdownify
from IPython.display import display, Markdown

# 이전에 확인한 이코노미톡뉴스 기사
article_url = "https://www.economytalk.kr/news/articleView.html?idxno=425107"

# 기사 페이지 가져오기
article_response = requests.get(
    article_url,
    timeout=30,
)
article_response.raise_for_status()

# HTML 읽기
soup = BeautifulSoup(article_response.content, "html.parser")

# 기사 제목과 본문 찾기
title_tag = soup.select_one("h1")
article_tag = soup.select_one("#article-view-content-div")

if article_tag is None:
    raise ValueError("기사 본문을 찾지 못했습니다.")

title = title_tag.get_text(" ", strip=True) if title_tag else "제목 없음"

# 본문을 마크다운으로 변환
article_markdown = markdownify(
    str(article_tag),
    heading_style="ATX",
    table_infer_header=True,
)

print("제목:", title)
display(Markdown(article_markdown))

제목: [주목 e-맞수] "영업익 늘었지만..." LG생건 vs 아모레퍼시픽…"中·음료 부진은 부담, Q4 턴어라운드?"


[김지현 기자 @이코노미톡뉴스] LG생활건강(이하 LG생건)이 2026년 3분기 뷰티 부문의 적자 폭을 줄이며 전년 동기 대비 50%가 넘는 영업이익 증가율을 기록했으나, 시장 기대치를 크게 밑도는 ‘어닝 쇼크’ 성적표를 받아들었다. 만성 적자 늪에 빠졌던 북미 자회사 에이본(The Avon Company)을 매각하며 재무 불확실성을 도려내는 결단을 내렸지만, 중국 시장의 구조적 침체와 캐시카우(현금창출원)였던 음료 부문의 수익성 악화가 겹치며 성장 동력이 둔화됐다는 평가다. 특히 같은 기간 아모레퍼시픽이 두 자릿수 영업이익률을 회복하고, 후발주자인 에이피알(APR)이 화장품 매출과 전사 영업이익에서 모두 LG생건을 추월하면서 국내 뷰티 업계의 지형도 변화가 한층 뚜렷해지고 있다.

**LG생건: 영업익 증가는 기저효과…"화장품 적자 지속·음료 수익성↓"**

---

금융감독원 전자공시시스템에 따르면 LG생건의 2026년 3분기 연결 기준 매출액은 1조 5,774억 원으로 전년 동기 대비 0.2% 감소했고, 영업이익은 704억 원으로 52.4% 증가했다. 표면적으로는 영업이익이 큰 폭으로 반등한 것처럼 보이지만, 이는 지난해 3분기 기저효과에 따른 결과로 증권가 컨센서스(전망치 평균 약 1,013억 원)를 30.5% 밑돈 수치다.

![LG생활건강은 2026년 3분기 영업이익이 전년 동기 대비 52.4% 증가한 704억 원을 기록하며 외형상 실적 개선에 성공했지만, 시장 기대치에는 크게 못 미치는 성적표를 받아들었다. 매출은 1조 5,774억 원으로 소폭 감소했고, 영업이익은 증권가 컨센서스를 약 30% 하회했다. [이미지=e톡뉴스 AI]](https://cdn.economytalk.kr/news/photo/202611/425107_233847_4527.png)

LG생활건강은 2026년 3분기 영업이익이 전년 동기 대비 52.4% 증가한 704억 원을 기록하며 외형상 실적 개선에 성공했지만, 시장 기대치에는 크게 못 미치는 성적표를 받아들었다. 매출은 1조 5,774억 원으로 소폭 감소했고, 영업이익은 증권가 컨센서스를 약 30% 하회했다. [이미지=e톡뉴스 AI]




직전 분기(2026년 2분기)와 비교하면 수익성 둔화는 더욱 두드러진다. 전분기 대비 매출은 4.8%, 영업이익은 31.5% 줄어들었으며 전사 영업이익률은 2분기 6.2%에서 3분기 4.5%로 하락했다. 여기에 북미 자회사 에이본 매각(9월부터 연결 제외)에 따른 일회성 처분손실 585억 원이 영업외비용으로 반영되면서 법인세비용차감전계속사업이익은 33억 원으로 전년 동기 대비 89.8% 급감했고, 당기순이익 역시 128억 원으로 45.2% 줄어들었다. 이 같은 실망감에 실적 발표 이튿날인 11월 4일 유가증권시장에서 LG생건 주가는 전 거래일 대비 11.83%(3만 1,000원) 급락한 23만 1,000원에 거래를 마쳤다

**아모레는 ‘턴어라운드’, 에이피알엔 ‘화장품 매출 역전’ 허용**

---

이번 3분기 실적에서 가장 뼈아픈 대목은 경쟁사들과의 수익성 및 본업 경쟁력 격차다. 글로벌 K-뷰티 호황 속에서 서구권 안착에 성공한 기업들과 구조조정 진통을 겪고 있는 LG생건의 성적표가 극명하게 엇갈렸다.

< K-뷰티 빅4의 Q3 실적 >

| **기업명** | **Q3 연결 매출(YoY)** | **Q3 영업익(YoY)** | **영업이익률** | **Q3 당기순익(YoY)** |
| --- | --- | --- | --- | --- |
| **LG생활건강** | 1조 5,774억 (-0.2%) | 704억 (+52.4%) | 4.5% | 128억 (-45.2%) |
| **아모레퍼시픽** | 1조 1,295 (+11.1%) | 1,152억 (+25.4%) | 10.2% | 585억 (-14.3%) |
| **에이피알(APR)** | 7,731억 (+100.3%) | 1,606억 (+68.9%) | 20.8% | 1,102억 (+47.6%) |
| **애경산업** | 1,904억 (+12.5%) | -33억 (적자전환) | -1.7% | -98억 (적자전환) |

![‘듀얼 리페어 리프트 M.D. 크림 미스트’. [사진=에이피 뷰티(AP BEAUTY) 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233846_4026.jpg)


‘듀얼 리페어 리프트 M.D. 크림 미스트’. [사진=에이피 뷰티(AP BEAUTY) 제공]

전체 외형에서는 음료와 생활용품을 거느린 LG생건이 1조 5,774억 원으로 1위를 지켰으나, 내실에서는 완전히 밀려났다. 아모레퍼시픽은 3분기 연결 매출 1조 1,295억 원(+11.1%), 영업이익 1,152억 원(+25.4%)을 기록하며 영업이익률 10.2%를 달성했다. 국내 사업 영업이익이 846억 원으로 42% 급증한 데다, 미주와 일본에서 에스트라·코스알엑스(COSRX)·설화수·헤라 등이 고성장하며 해외 영업이익도 485억 원(+14%)을 거뒀다.

더욱 충격적인 변화는 신흥 강자 에이피알(APR)의 약진이다. 에이피알은 해외 매출 비중이 94%까지 확대되며 3분기 매출 7,731억 원(+100.3%), 영업이익 1,606억 원(+68.9%, 영업이익률 20.8%)으로 창사 이래 최대 분기 매출을 새로 썼다. 특히 에이피알의 화장품·뷰티 단일 부문 매출은 6,823억 원(+150.6%)으로, LG생건의 전체 뷰티 부문 매출(6,608억 원)을 처음으로 앞질렀다. 전사 영업이익에서도 에이피알은 LG생건의 2.3배에 달하는 이익을 내며 K-뷰티 수익성 1위 자리를 차지했다.

![지난 9월 24~27일 멕시코 멕시코시티에서 열린 K-엑스포에 마련된 더페이스샵 홍보부스에 많은 인파가 몰렸다. [사진=LG H&H Co., Ltd 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233844_3258.jpg)

지난 9월 24~27일 멕시코 멕시코시티에서 열린 K-엑스포에 마련된 더페이스샵 홍보부스에 많은 인파가 몰렸다. [사진=LG H&H Co., Ltd 제공]

**LG생건: ‘에이본 손절·국내 효율화’ 빛났지만 ‘미·중 적자·음료 부진’**

---

LG생건의 3분기 실적을 사업부별로 살펴보면 구조조정 효과가 나타나기 시작했다는 점은 긍정적이다. 뷰티 부문 매출은 6,608억 원(-0.3% YoY)으로 전년 수준을 유지했으며, 영업손실은 지난해 486억 원에서 109억 원으로 크게 줄어 적자 규모를 377억 원 축소했다. 국내 백화점과 할인점 중심의 비효율 오프라인 채널을 정리하고 온라인·MBS(멀티브랜드숍)·면세 중심으로 영업 구조를 재편해 고정비를 낮춘 효과다. 특히 면세 매출이 145.5% 급증(925억 원)하며 주력 브랜드 더후가 성장세로 전환했고, 닥터그루트·도미나스·유시몰(오랄 뷰티)·VDL 등도 고성장을 이어갔으며, 만성 적자 사업이었던 북미 에이본 매각까지 마무리하면서 장기적인 손익 개선 기반을 마련했다.

전략 브랜드 육성 성과도 가시화되고 있다. 3분기 북미 전체 매출은 에이본 연결 제외 영향으로 1,355억 원(-12.5%)을 기록했으나, 에이본을 제외한 북미 자체 브랜드 매출은 약 5% 성장했다. 주력 헤어케어 브랜드 닥터그루트가 지난 8월 북미 세포라 오프라인 매장 400여 곳에 입점한 데 이어, 더페이스샵이 미국 월마트(약 1,600개 매장)와 얼타(ULTA, 약 500개 매장)에, 인디 색조 브랜드 힌스(hince)가 얼타(약 400개 매장)에 잇따라 입점하며 북미 리테일 접점을 넓혔다. 여기에 힌스를 운영하는 자회사 비바웨이브의 잔여 지분 25%를 콜옵션 행사로 추가 매입해 지분 100% 체제를 구축했고, 생활용품(HDB) 부문 역시 ‘피지 모락셀라(세탁세제)’, ‘46cm(체취케어)’, ‘홈스타(주거용품)’ 등 히어로 제품이 실적을 견인하며 매출 4,192억 원(+3.8%), 영업이익 310억 원(-3.1%)을 기록했다.

![최근 LG생활건강의 대표 K-뷰티 브랜드 ‘더페이스샵’이 미국 대형 화장품 유통업체 ‘얼타 뷰티(Ulta Beauty)’ 온·오프라인에 입점했다. (사진은) 더페이스샵 미감수 제품 라인업. [사진=LG Life & Health 제공]](https://cdn.economytalk.kr/news/photo/202611/425107_233845_3642.jpg)

최근 LG생활건강의 대표 K-뷰티 브랜드 ‘더페이스샵’이 미국 대형 화장품 유통업체 ‘얼타 뷰티(Ulta Beauty)’ 온·오프라인에 입점했다. (사진은) 더페이스샵 미감수 제품 라인업. [사진=LG Life & Health 제공]

반면 해외 핵심 시장의 적자 지속과 비화장품 사업 약세는 여전히 해결해야 할 과제다. 전체 해외 매출은 4,323억 원으로 12.5% 감소했으며, 특히 중국 매출이 온라인 사업 구조 재편 여파로 21.1% 급감한 1,151억 원에 그쳤다. 더후 중심의 중국 사업이 침체를 벗어나지 못하면서 고정비 부담에 따른 적자가 이어지고 있고, 북미 역시 신규 채널 입점 초기 단계의 마케팅 비용 투입이 늘어나며 외형 성장이 수익성으로 직결되지 못했다. 더욱이 화장품 부진을 방어하던 음료(리프레시먼트) 사업도 매출 4,974억 원(-3.1%), 영업이익 503억 원(-19.9%)으로 크게 둔화됐다. 내수 소비 침체와 원부자재 가격 상승, 제로 탄산 판촉 경쟁 심화가 겹치면서 LG생건은 중국 회복과 북미 안착, 음료 수익성 방어라는 세 가지 과제를 동시에 해결해야 하는 상황에 놓여 있다.

< 증권가 Q4 향후 전망 >

| **증권사** | **애널리스트** | **투자의견** | **목표주가 변동** |
| --- | --- | --- | --- |
| **메리츠증권** | 박신애·김건우 | 매수 유지 | 37만 → 32만 (-13.5%) |
| **한국투자증권** | 김명주 | 매수 유지 | 34만 → 31만 (-8.8%) |
| **키움증권** | 손예빈 | 매수→중립 | 33만 → 28만 (-15.2%) |
| **하나증권** | 박은정 | 매수→중립 | 32만 → 27만 (-15.6%) |
| **LS증권** | 오린아 | 매수→보유 | 35만 → 26만 (-25.7%) |
| **DB증권** | 허제나 | 매수 유지 | 36만 유지 |
| **한화투자증권** | 한유정 | 매수 유지 | 35만 유지 |
| **신한투자증권** | 박현진 | 매수 유지 | 33만 5,000원 유지 |

**“본격적인 턴어라운드는 2027년 이후?”**

---

3분기 어닝 쇼크 이후 주요 증권사들은 목표주가를 일제히 낮추며 당분간 보수적인 접근이 필요하다는 의견을 내놓고 있다. 한국투자증권과 메리츠증권은 생활용품을 제외한 뷰티·음료 사업의 회복 속도가 예상보다 더디고 전사 성장을 주도할 단기 모멘텀이 부족하다고 평가했다. 키움증권과 하나증권 역시 중국 사업 적자가 이어지는 가운데 북미 오프라인 유통망 확장에 따른 초기 비용 부담이 불가피해, 미국과 중국 시장에서 화장품 사업의 실질적인 수익성 개선이 확인되기 전까지는 시간이 필요하다며 투자의견을 ‘중립’으로 하향했다. 메리츠증권은 4분기 영업이익을 69억 원, 한화투자증권은 4분기 매출 1조 5,071억 원(+2.4%)과 영업이익 134억 원(흑자전환)으로 추정하며 계절적 비수기와 구조조정 비용 영향으로 4분기 이익 규모는 제한적일 것으로 내다봤다.

다만 체질 개선 방향성 자체는 긍정적이라는 평가도 공존한다. 한화투자증권, 신한투자증권, DB증권 등은 면세 채널의 수익 구조 개선과 에이본 매각을 통한 북미 사업 단순화, 그리고 닥터그루트·더페이스샵·힌스를 앞세운 북미·일본 채널 확장이 중장기 실적 반등의 발판이 될 것이라며 목표주가를 유지했다.

결국 시장의 관심은 2027년 이후 실질적인 턴어라운드 가능성에 집중되고 있다. LG생건이 K-뷰티 선두권 지위를 회복하기 위해서는 에이본 매각 이후 가벼워진 북미 법인에서 닥터그루트와 힌스가 초기 마케팅비를 상쇄하는 핵심 성장동력으로 자리 잡고, 중국 온라인 채널 재편을 조속히 마무리해 화장품 본업의 안정적인 흑자 구조를 구축해야 한다. 증권가가 공통적으로 제시하는 핵심 과제는 단순한 ‘적자 축소’를 넘어선 ‘지속 가능한 외형 성장과 수익성 회복’이며, 그 성과가 본격적으로 확인되는 시점은 2027년 상반기 이후가 될 전망이다.

In [7]:
import time

# 언론사에서 자주 쓰는 본문 영역
article_selectors = [
    "#article-view-content-div",
    "#article-view-content",
    "#articleBody",
    "#article-body",
    "#newsContent",
    "#newsct_article",
    ".article-body",
    ".article-content",
    ".news_view",
    "article",
]

# 기사별 결과를 담을 리스트
article_results = []

for index, row in news_df.iterrows():
    article_url = row["original_url"]

    # 현재 기사의 결과를 기본값으로 준비
    result = {
        "title": row["title"],
        "original_url": article_url,
        "markdown": None,
        "selector": None,
        "status": "실패",
        "error": None,
    }

    try:
        if not article_url:
            raise ValueError("원문 주소가 없습니다.")

        # 기사 페이지 요청
        article_response = requests.get(article_url, timeout=30)
        article_response.raise_for_status()

        soup = BeautifulSoup(article_response.content, "html.parser")

        # 본문 영역을 순서대로 찾기
        article_tag = None

        for selector in article_selectors:
            candidate = soup.select_one(selector)

            if candidate and len(candidate.get_text(" ", strip=True)) >= 100:
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            raise ValueError("본문 영역을 찾지 못했습니다.")

        # HTML 본문을 마크다운으로 변환
        result["markdown"] = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 본문 후보는 찾았지만 실제 내용 확인이 필요함
        result["status"] = "확인 필요"

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except ValueError as error:
        result["error"] = str(error)

    article_results.append(result)

    print(f"[{index + 1}/{len(news_df)}] {result['status']} · {row['title']}")

    # 사이트에 요청을 연속으로 보내지 않도록 잠시 대기
    time.sleep(1)

# 결과를 표로 변환
articles_df = pd.DataFrame(article_results)

display(
    articles_df[["title", "status", "selector", "error"]]
)

[1/10] 실패 · 국회 입법박람회 개막...로봇·수소차·누리호 등 첨단기술 한자리에
[2/10] 확인 필요 · [기획] K뷰티 수출 신기록에도 화장품주 약세
[3/10] 확인 필요 · '보고 체험하고 제안하는' 2026 국회 입법박람회 개막


KeyboardInterrupt: 

In [ ]:
# 본문을 찾지 못한 기사만 보기
failed_articles = articles_df[
    articles_df["status"] == "실패"
]

display(
    failed_articles[["title", "original_url", "error"]]
)

,title,original_url,error
0,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,본문 영역을 찾지 못했습니다.
8,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,본문 영역을 찾지 못했습니다.
9,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/2026110550...,본문 영역을 찾지 못했습니다.


In [ ]:
# 아시아타임즈 기사 주소
article_url = "https://www.asiatime.co.kr/article/20261105500303"

# 페이지 가져오기
article_response = requests.get(article_url, timeout=30)
article_response.raise_for_status()

soup = BeautifulSoup(article_response.content, "html.parser")

# 아시아타임즈의 본문 영역
article_tag = soup.select_one("#articleContent")

if article_tag is None:
    print("requests로 받은 HTML에서 본문을 찾지 못했습니다.")
else:
    article_markdown = markdownify(
        str(article_tag),
        heading_style="ATX",
        table_infer_header=True,
    )

    print("본문 영역을 찾았습니다!")
    display(Markdown(article_markdown))

본문 영역을 찾았습니다!


advertisement

advertisement

잔디광장 로봇 뛰는 ‘입법박람회’…시민 정책·신기술 한마당

본회의장 ‘조작기소 특검’ 충돌…野 전면 필리버스터 전운

국민의힘 “방탄·악법 박람회…국회 앞 맞불 여론전까지 검토”

[아시아타임즈=장유림 기자] 서울 여의도 국회의사당 잔디광장에서 미래 신기술과 시민 정책이 어우러진 ‘입법박람회’가 개막해 활기를 띠었다. 반면 본관 안 본회의장 앞은 ‘조작기소 특검법’ 상정을 둘러싼 여야의 극한 대치와 국민의힘의 무제한 토론(필리버스터) 예고로 무거운 전운이 감돌며 국회 안팎의 풍경은 극명한 대조를 이뤘다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500301.jpg)


5일 서울 여의도 국회의사당에서 ‘2026 국회 입법박람회’가열렸다.(사진=장유림 기자)

5일 오전 개막한 ‘2026 국회 입법박람회’는 국민의 아이디어를 정책과 법률로 잇겠다는 취지에 걸맞게 개막 첫날부터 시민들의 큰 호응을 얻었다. 개막식에서는 타악그룹 ‘타고’의 힘찬 모둠북 연주가 울려 퍼졌고, 어린이 2명이 낭독한 ‘국민희망입법제안문’을 현대자동차그룹의 4족 보행 로봇 ‘스팟(Spot)’이 조정식 국회의장에게 전달하는 이색 광경도 펼쳐졌다. 경제·문화·미래·입법 등 4개 권역으로 꾸려져 88개 기관·기업·단체가 참여한 잔디광장 부스에는 이른 시간부터 관람객의 발길이 이어졌다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500320.png)


5일 국회 중앙잔디광장에서 열린 2026 국회 입법박람회 경제존 안 아모레퍼시픽 부스에서 피부색 측정을 위해 시민들이 대기하고 있다.(사진=장유림기자)

특히 기업들이 실물 메모리 반도체와 휴머노이드 로봇 등을 선보인 경제존 부스는 시민들의 발걸음을 멈춰 세웠다. 4족 보행 로봇 주위로 시민들이 모여들었고 아모레퍼시픽 부스 앞에서는 최신 기술로 피부색을 측정하고 맞춤형 뷰티 솔루션을 체험하려는 관람객들로 긴 줄이 늘어섰다.

잔디광장 곳곳에서는 시민들이 자유롭게 부스를 오가며 체험에 참여했고, 아이들의 웃음소리가 끊이지 않았다. 행사에 참여한 이모(50) 씨는 “국회라고 하면 늘 엄숙하고 정쟁으로 싸우기만 하는 곳인 줄 알았는데, 바로 앞마당에서 활기찬 축제가 열리니 국회가 한결 친근하게 느껴진다”고 소감을 전했다.

advertisement

advertisement

미래존에서 정책 제안에 나선 이윤아(17·설월여고) 양도 “작년에 나갔던 정책 대회는 발표 시간이 30분밖에 안 돼 아쉬웠는데, 넓은 잔디광장에서 하루 종일 시민들에게 직접 정책을 설명하고 자유롭게 제안할 수 있어 뜻깊다”며 웃었다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500302.jpg)


국민의힘 장동혁 대표와 정점식 원내대표가 5일 국회에서 열린 의원총회에서 민주당 규탄 구호를 외치고 있다.(사진=연합뉴스)

같은 시각 국회 본관은 사정이 달랐다. 잔디광장의 ‘국민입법제안소’에는 민생을 개선해 달라는 시민들의 손글씨 제안서가 쌓여갔지만, 본관에서는 여야의 입법 충돌로 의사일정이 멈춰 섰다.

이날 오전 여야 원내대표는 윤석열 정부 당시 수사기관의 표적 수사 의혹을 파헤치는 특검법 상정을 두고 막판 담판을 벌였으나 합의점을 찾지 못했다. 더불어민주당(이하 민주당)이 특검법을 비롯해 민주유공자법, 초광역특구법 등 6개 안건을 이날 본회의에서 처리하겠다고 나서자 국민의힘은 상정되는 모든 법안에 필리버스터를 걸겠다며 맞섰다. 여야가 이견을 좁히지 못하면서 당초 오후 2시로 예정됐던 본회의는 오후 3시로 1시간 미뤄졌다.

국민의힘은 경내의 축제 분위기를 겨냥해 민주당을 비판했다. 장동혁 국민의힘 대표는 이날 오전 당 최고위원회의(이하 최고위)에서 “악법만 통과시키면서 입법박람회는 대체 왜 하는 것이냐”며 “입법박람회를 열어놓고 오늘 본회의에 조작기소 특검법을 상정한다면 국민들이 뭐라고 하겠느냐”고 비판했다. 이어 “박람회를 열고 이런 악법에 대해 야당에 필리버스터를 하라고 종용하는 꼴 아니냐”며 “강행한다면 박람회를 이재명 공소취소 박람회로 만드는 것”이라고 강조했다.

국민의힘 지도부는 장외 여론전까지 검토하고 있다. 최보윤 국민의힘 수석대변인은 이날 오전 최고위 직후 기자들과 만난 자리에서 “입법박람회가 진행 중인데, 국민들 보시기에 정말 ‘방탄 입법박람회’, ‘위헌 입법박람회’, ‘악법 입법박람회’라고 볼 수밖에 없다”고 꼬집었다. 이어 “입법박람회가 열리는 밖과, 국회 안에서 필리버스터를 진행할 수밖에 없는 굉장히 대비되는 상황”이라며 “국회 앞에서 별도로 목소리를 낼 수 있는 방안까지 검토하고 있다”고 밝혔다.

In [ ]:
# 아시아타임즈 기사 주소
article_url = "https://www.asiatime.co.kr/article/20261105500303"

# 기존 선택자 목록에 아시아타임즈 본문 위치 추가
if "#articleContent" not in article_selectors:
    article_selectors.insert(0, "#articleContent")

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML에서 본문 찾기
    soup = BeautifulSoup(response.content, "html.parser")
    article_tag = soup.select_one("#articleContent")

    if article_tag is None:
        print("본문을 찾지 못했습니다.")
    else:
        # 본문을 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 articles_df에서 해당 기사 찾기
        article_mask = articles_df["original_url"].str.contains(
            "asiatime.co.kr/article/20261105500303",
            regex=False,
            na=False,
        )

        # 해당 기사의 결과 업데이트
        articles_df.loc[article_mask, "markdown"] = article_markdown
        articles_df.loc[article_mask, "selector"] = "#articleContent"
        articles_df.loc[article_mask, "status"] = "확인 필요"
        articles_df.loc[article_mask, "error"] = None

        print("본문을 찾고 기존 결과를 업데이트했습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("페이지 요청에 실패했습니다:", error)

본문을 찾고 기존 결과를 업데이트했습니다!


advertisement

advertisement

잔디광장 로봇 뛰는 ‘입법박람회’…시민 정책·신기술 한마당

본회의장 ‘조작기소 특검’ 충돌…野 전면 필리버스터 전운

국민의힘 “방탄·악법 박람회…국회 앞 맞불 여론전까지 검토”

[아시아타임즈=장유림 기자] 서울 여의도 국회의사당 잔디광장에서 미래 신기술과 시민 정책이 어우러진 ‘입법박람회’가 개막해 활기를 띠었다. 반면 본관 안 본회의장 앞은 ‘조작기소 특검법’ 상정을 둘러싼 여야의 극한 대치와 국민의힘의 무제한 토론(필리버스터) 예고로 무거운 전운이 감돌며 국회 안팎의 풍경은 극명한 대조를 이뤘다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500301.jpg)


5일 서울 여의도 국회의사당에서 ‘2026 국회 입법박람회’가열렸다.(사진=장유림 기자)

5일 오전 개막한 ‘2026 국회 입법박람회’는 국민의 아이디어를 정책과 법률로 잇겠다는 취지에 걸맞게 개막 첫날부터 시민들의 큰 호응을 얻었다. 개막식에서는 타악그룹 ‘타고’의 힘찬 모둠북 연주가 울려 퍼졌고, 어린이 2명이 낭독한 ‘국민희망입법제안문’을 현대자동차그룹의 4족 보행 로봇 ‘스팟(Spot)’이 조정식 국회의장에게 전달하는 이색 광경도 펼쳐졌다. 경제·문화·미래·입법 등 4개 권역으로 꾸려져 88개 기관·기업·단체가 참여한 잔디광장 부스에는 이른 시간부터 관람객의 발길이 이어졌다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500320.png)


5일 국회 중앙잔디광장에서 열린 2026 국회 입법박람회 경제존 안 아모레퍼시픽 부스에서 피부색 측정을 위해 시민들이 대기하고 있다.(사진=장유림기자)

특히 기업들이 실물 메모리 반도체와 휴머노이드 로봇 등을 선보인 경제존 부스는 시민들의 발걸음을 멈춰 세웠다. 4족 보행 로봇 주위로 시민들이 모여들었고 아모레퍼시픽 부스 앞에서는 최신 기술로 피부색을 측정하고 맞춤형 뷰티 솔루션을 체험하려는 관람객들로 긴 줄이 늘어섰다.

잔디광장 곳곳에서는 시민들이 자유롭게 부스를 오가며 체험에 참여했고, 아이들의 웃음소리가 끊이지 않았다. 행사에 참여한 이모(50) 씨는 “국회라고 하면 늘 엄숙하고 정쟁으로 싸우기만 하는 곳인 줄 알았는데, 바로 앞마당에서 활기찬 축제가 열리니 국회가 한결 친근하게 느껴진다”고 소감을 전했다.

advertisement

advertisement

미래존에서 정책 제안에 나선 이윤아(17·설월여고) 양도 “작년에 나갔던 정책 대회는 발표 시간이 30분밖에 안 돼 아쉬웠는데, 넓은 잔디광장에서 하루 종일 시민들에게 직접 정책을 설명하고 자유롭게 제안할 수 있어 뜻깊다”며 웃었다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500302.jpg)


국민의힘 장동혁 대표와 정점식 원내대표가 5일 국회에서 열린 의원총회에서 민주당 규탄 구호를 외치고 있다.(사진=연합뉴스)

같은 시각 국회 본관은 사정이 달랐다. 잔디광장의 ‘국민입법제안소’에는 민생을 개선해 달라는 시민들의 손글씨 제안서가 쌓여갔지만, 본관에서는 여야의 입법 충돌로 의사일정이 멈춰 섰다.

이날 오전 여야 원내대표는 윤석열 정부 당시 수사기관의 표적 수사 의혹을 파헤치는 특검법 상정을 두고 막판 담판을 벌였으나 합의점을 찾지 못했다. 더불어민주당(이하 민주당)이 특검법을 비롯해 민주유공자법, 초광역특구법 등 6개 안건을 이날 본회의에서 처리하겠다고 나서자 국민의힘은 상정되는 모든 법안에 필리버스터를 걸겠다며 맞섰다. 여야가 이견을 좁히지 못하면서 당초 오후 2시로 예정됐던 본회의는 오후 3시로 1시간 미뤄졌다.

국민의힘은 경내의 축제 분위기를 겨냥해 민주당을 비판했다. 장동혁 국민의힘 대표는 이날 오전 당 최고위원회의(이하 최고위)에서 “악법만 통과시키면서 입법박람회는 대체 왜 하는 것이냐”며 “입법박람회를 열어놓고 오늘 본회의에 조작기소 특검법을 상정한다면 국민들이 뭐라고 하겠느냐”고 비판했다. 이어 “박람회를 열고 이런 악법에 대해 야당에 필리버스터를 하라고 종용하는 꼴 아니냐”며 “강행한다면 박람회를 이재명 공소취소 박람회로 만드는 것”이라고 강조했다.

국민의힘 지도부는 장외 여론전까지 검토하고 있다. 최보윤 국민의힘 수석대변인은 이날 오전 최고위 직후 기자들과 만난 자리에서 “입법박람회가 진행 중인데, 국민들 보시기에 정말 ‘방탄 입법박람회’, ‘위헌 입법박람회’, ‘악법 입법박람회’라고 볼 수밖에 없다”고 꼬집었다. 이어 “입법박람회가 열리는 밖과, 국회 안에서 필리버스터를 진행할 수밖에 없는 굉장히 대비되는 상황”이라며 “국회 앞에서 별도로 목소리를 낼 수 있는 방안까지 검토하고 있다”고 밝혔다.

In [ ]:
# 아직 본문을 찾지 못한 기사만 확인
failed_articles = articles_df[
    articles_df["status"] == "실패"
]

print(f"아직 확인이 필요한 기사: {len(failed_articles)}건")

display(
    failed_articles[["title", "original_url", "error"]]
)

아직 확인이 필요한 기사: 2건


,title,original_url,error
8,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,본문 영역을 찾지 못했습니다.
9,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/2026110550...,본문 영역을 찾지 못했습니다.


In [ ]:
# 스카이데일리의 본문 선택자 추가
if "#gisaview" not in article_selectors:
    article_selectors.insert(0, "#gisaview")

# 실패했던 8번 기사 주소 가져오기
article_url = articles_df.loc[8, "original_url"]

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML 읽기
    soup = BeautifulSoup(response.content, "html.parser")

    # 스카이데일리의 본문 영역 찾기
    article_tag = soup.select_one("#gisaview")

    if article_tag is None:
        print("본문 영역을 찾지 못했습니다.")
    else:
        # 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 결과 업데이트
        articles_df.at[8, "markdown"] = article_markdown
        articles_df.at[8, "selector"] = "#gisaview"
        articles_df.at[8, "status"] = "확인 필요"
        articles_df.at[8, "error"] = None

        print("스카이데일리 본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


본문 영역을 찾지 못했습니다.


In [ ]:
# 스카이데일리 기사 응답 확인
print("요청한 주소:", article_url)
print("실제로 받은 주소:", response.url)
print("HTTP 상태:", response.status_code)
print("서버가 알려준 인코딩:", response.encoding)

# HTML에 본문 영역 이름이 있는지 확인
print("'gisaview' 포함 여부:", b"gisaview" in response.content)

# 서버가 보낸 HTML 앞부분 확인
print("\n받은 페이지 앞부분:")
print(response.text[:500])

요청한 주소: https://m.skyedaily.com/news_view.html?ID=314172
실제로 받은 주소: https://m.skyedaily.com/news_view.html?ID=314172
HTTP 상태: 200
서버가 알려준 인코딩: ISO-8859-1
'gisaview' 포함 여부: False

받은 페이지 앞부분:




<!DOCTYPE html>
<html>
<head>
    <title>ìë²ë°ëíì ë±ì¥í ì¬ì¡±ë¡ë´</title>    
    <meta http-equiv="content-type" content="text/html; charset=utf-8">
    

	<meta name="image" content="https://pds.skyedaily.com/top_image/202611/314172_p.JPG" />
	<meta property="og:image" content="https://pds.skyedaily.com/top_image/202611/314172_p.JPG" />

	<link rel="image_src" href="https://pds.skyedaily.com/top_image/202611/314172_p.JPG">

		


	<meta name="title" conte


In [ ]:
# 같은 기사의 PC 페이지 주소
desktop_url = "https://www.skyedaily.com/news/news_view.html?ID=314172"

try:
    # PC 페이지 요청
    response = requests.get(desktop_url, timeout=30)
    response.raise_for_status()

    # HTML에 표시된 UTF-8 인코딩 적용
    response.encoding = "utf-8"

    # HTML 읽기
    soup = BeautifulSoup(response.text, "html.parser")

    # PC 페이지의 본문 영역
    article_tag = soup.select_one("#gisaview")

    if article_tag is None:
        print("PC 페이지에서도 본문을 찾지 못했습니다.")
    else:
        # 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 8번 기사의 결과 업데이트
        articles_df.at[8, "markdown"] = article_markdown
        articles_df.at[8, "selector"] = "#gisaview"
        articles_df.at[8, "status"] = "확인 필요"
        articles_df.at[8, "error"] = None

        print("본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

본문을 찾았습니다!


![](https://pds.skyedaily.com/news_data2025_2/20261105140309_vsgeskvc.jpg)


�� ������ ��ȸ������ 5�� ���� ���ǵ� ��ȸ �ܵ��忡�� ���� 2026 ��ȸ �Թ��ڶ�ȸ���� �������� �κ����κ��� ��������Թ����ȹ��� �ް� �ִ�. �̹� ���� ������ ��Ҹ��� ���� �Թ��� ��å���� �����ϰ� ����� ������ �޹�ħ�ϱ� ���� ���õƴ�. ��������ȭ���̷����Թ� �� 4�� ���� ������ ��ȸ�� ����, �������, ���, �ùδ�ü �� 88�� ������������ü�� �����Ѵ�. �ڹ̳� ���� �Ͻ�ī�̵��ϸ�

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_huxmdrdu.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_ezybkevo.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_mlxzfocz.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_qnkdgmwp.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_zukxvwal.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_safmuibs.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_noknoktv.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_zyfvuzsx.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_xohyffvs.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_nfycymip.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_bwvolkls.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_gtonwpui.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_xnusxnxe.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_nahcnozt.jpg)


�� ��ȸ�� �ϴ� ������ ��ȸ����.

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_vsaqavvt.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_hizwzquu.jpg)


�� ����ϴ� �Ѻ��� ���Ҿ���ִ� ������ǥ.

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_lkmgcrto.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_lyghvzii.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_zfumpgat.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_bxfewedm.jpg)


�� ������ �ν� -�Ƹ��۽���

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_kgnxiend.jpg)


�� ������ �ν�-LG

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_albgtkla.jpg)


�� ������ �ν�- �ѱ�������������ȸ

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_vtdizqyh.jpg)


�� ������ �ν�- KAI

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_izhrxhcq.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_gawncqyf.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140625_yyyoenwm.jpg)


�� ������ �ν�- ��ȭ . �ڹ̳� ���� �Ͻ�ī�̵��ϸ�

In [ ]:
# PC 페이지에서 받은 원본 바이트
html_bytes = response.content

# 인코딩별로 읽기를 시도하고 본문 일부 확인
for encoding in ["utf-8", "cp949", "euc-kr"]:
    print(f"\n--- {encoding} ---")

    try:
        # 오류를 대체하지 않고 정확히 읽을 수 있는지 확인
        decoded_html = html_bytes.decode(encoding)

        test_soup = BeautifulSoup(decoded_html, "html.parser")
        test_article = test_soup.select_one("#gisaview")

        if test_article is None:
            print("본문 영역을 찾지 못했습니다.")
        else:
            print(test_article.get_text(" ", strip=True)[:200])

    except UnicodeDecodeError:
        print("이 인코딩으로 읽을 수 없습니다.")


--- utf-8 ---
이 인코딩으로 읽을 수 없습니다.

--- cp949 ---
▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 국회 입법박람회에서 사족보행 로봇으로부터 국민희망입법제안문을 받고 있다. 이번 행사는 국민의 목소리를 실제 입법과 정책으로 연결하고 기업의 혁신을 뒷받침하기 위해 마련됐다. 경제·문화·미래·입법 등 4개 전시 공간에 국회와 정부, 공공기관, 기업, 시민단체 등 88개 기관·기업·단체가 

--- euc-kr ---
▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 국회 입법박람회에서 사족보행 로봇으로부터 국민희망입법제안문을 받고 있다. 이번 행사는 국민의 목소리를 실제 입법과 정책으로 연결하고 기업의 혁신을 뒷받침하기 위해 마련됐다. 경제·문화·미래·입법 등 4개 전시 공간에 국회와 정부, 공공기관, 기업, 시민단체 등 88개 기관·기업·단체가 


In [ ]:
# PC 페이지의 원본 데이터를 cp949로 읽기
decoded_html = response.content.decode("cp949")

# HTML에서 본문 찾기
soup = BeautifulSoup(decoded_html, "html.parser")
article_tag = soup.select_one("#gisaview")

if article_tag is None:
    print("본문 영역을 찾지 못했습니다.")
else:
    # 본문을 마크다운으로 변환
    article_markdown = markdownify(
        str(article_tag),
        heading_style="ATX",
        table_infer_header=True,
    )

    # 8번 기사의 깨진 결과를 정상 내용으로 교체
    articles_df.at[8, "markdown"] = article_markdown
    articles_df.at[8, "selector"] = "#gisaview"
    articles_df.at[8, "status"] = "확인 필요"
    articles_df.at[8, "error"] = None

    print("한글을 정상적으로 읽었습니다!")
    display(Markdown(article_markdown))

한글을 정상적으로 읽었습니다!


![](https://pds.skyedaily.com/news_data2025_2/20261105140309_vsgeskvc.jpg)


▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 국회 입법박람회에서 사족보행 로봇으로부터 국민희망입법제안문을 받고 있다. 이번 행사는 국민의 목소리를 실제 입법과 정책으로 연결하고 기업의 혁신을 뒷받침하기 위해 마련됐다. 경제·문화·미래·입법 등 4개 전시 공간에 국회와 정부, 공공기관, 기업, 시민단체 등 88개 기관·기업·단체가 참여한다. 박미나 기자 ⓒ스카이데일리

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_huxmdrdu.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_ezybkevo.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_mlxzfocz.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_qnkdgmwp.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140508_zukxvwal.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_safmuibs.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_noknoktv.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_zyfvuzsx.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_xohyffvs.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_nfycymip.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_bwvolkls.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140509_gtonwpui.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_xnusxnxe.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_nahcnozt.jpg)


▲ 개회사 하는 조정식 국회의장.

![](https://pds.skyedaily.com/news_data2025_2/20261105140543_vsaqavvt.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_hizwzquu.jpg)


▲ 축사하는 한병도 더불어민주당 원내대표.

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_lkmgcrto.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_lyghvzii.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140544_zfumpgat.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_bxfewedm.jpg)


▲ 경제존 부스 -아모레퍼시픽

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_kgnxiend.jpg)


▲ 경제존 부스-LG

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_albgtkla.jpg)


▲ 경제존 부스- 한국여성경제인협회

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_vtdizqyh.jpg)


▲ 경제존 부스- KAI

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_izhrxhcq.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140624_gawncqyf.jpg)

![](https://pds.skyedaily.com/news_data2025_2/20261105140625_yyyoenwm.jpg)


▲ 경제존 부스- 한화 . 박미나 기자 ⓒ스카이데일리

In [ ]:
print(articles_df.loc[9, "original_url"])

http://www.metroseoul.co.kr/article/20261105500277


In [ ]:
# 메트로서울의 본문 선택자 추가
if ".article-txt-contents" not in article_selectors:
    article_selectors.insert(0, ".article-txt-contents")

# 실패했던 9번 기사 주소
article_url = articles_df.loc[9, "original_url"]

try:
    # 기사 페이지 요청
    response = requests.get(article_url, timeout=30)
    response.raise_for_status()

    # HTML 읽기
    soup = BeautifulSoup(response.content, "html.parser")

    # 메트로서울의 본문 영역 찾기
    article_tag = soup.select_one(".article-txt-contents")

    if article_tag is None:
        print("본문 영역을 찾지 못했습니다.")
    else:
        # 본문을 마크다운으로 변환
        article_markdown = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        # 기존 결과 업데이트
        articles_df.at[9, "markdown"] = article_markdown
        articles_df.at[9, "selector"] = ".article-txt-contents"
        articles_df.at[9, "status"] = "확인 필요"
        articles_df.at[9, "error"] = None

        print("메트로서울 본문을 찾았습니다!")
        display(Markdown(article_markdown))

except requests.RequestException as error:
    print("기사 페이지를 가져오지 못했습니다:", error)

메트로서울 본문을 찾았습니다!


![](//cdn.emetro.co.kr/data2/content/image/2026/11/05/.cache/512/20261105500278.jpg)


에이피 뷰티 '듀얼 리페어 리프트 M.D. 크림 미스트' /아모레퍼시픽.

아모레퍼시픽의 하이엔드 스킨케어 브랜드 에이피 뷰티가 스킨케어 핵심 성분을 미스트 제형으로 구현해 시술 후 피부 회복 수요 대응에 나선다.

에이피 뷰티는 전국 주요 백화점 및 온라인몰에서 '듀얼 리페어 리프트 M.D. 크림 미스트'를 출시한다고 5일 밝혔다. 브랜드 대표 제품인 '듀얼 리페어 리프트 크림'의 유효 성분과 기술력을 집약한 신제품이다.

특히 특수 관리 후 민감해진 피부 장벽을 얇은 액체 밴드 형태로 밀폐·보호하는 독자적 제제 공학 기술을 적용했다.

이와 함께 일반 연어 유래 PDRN보다 회복 효과가 2.1배 높은 고순도 228K PDRN, 엑소인, 고분자 티알에프 히알루론산 등을 배합해 피부 표피 깊숙이 수분을 공급하고 진정을 돕는다.

피부과 관리 기술에서 착안한 '리퀴드 밴디지 텍스처'를 적용해 크림 유화 입자를 모공 크기의 약 1800분의 1 수준으로 처방한 것도 특징이다. 초미세 입자를 피부 표면에 분사하면 얇은 '마이크로 리페어 보호막'이 자극받은 피부 수분 손실을 막고 물리적 보호막 역할을 수행한다. 실제 인체적용시험에서도 피부과 관리 후 사용 시 진정 효과가 65.5%, 수분량이 137% 개선됐다.

에이피 뷰티 관계자는 "화장품으로 구현하기 어려웠던 특수 관리 성분을 접목한 M.D. 라인을 통해, 일상 케어는 물론 피부과 관리 후 집중적인 장벽 수복까지 아우르는 고기능성 솔루션을 지속 제공하겠다"고 말했다.

In [ ]:
# 기사별 본문 수집 상태 확인
articles_df["본문_글자수"] = articles_df["markdown"].fillna("").str.len()

display(
    articles_df[
        ["title", "status", "selector", "본문_글자수", "error"]
    ]
)

print("전체 기사:", len(articles_df), "건")
print("본문이 있는 기사:", (articles_df["본문_글자수"] > 0).sum(), "건")
print("본문이 없는 기사:", (articles_df["본문_글자수"] == 0).sum(), "건")

,title,status,selector,본문_글자수,error
0,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",확인 필요,#articleContent,2486,NaN
1,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,확인 필요,#article-view-content-div,2055,NaN
2,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,확인 필요,#article-view-content-div,2346,NaN
3,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",확인 필요,article,5181,NaN
4,"이니스프리 모음재단, 제주 '모드락마당' 조성",확인 필요,#article-view-content-div,1664,NaN
5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,확인 필요,#articleBody,2653,NaN
6,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,확인 필요,.article-body,2443,NaN
7,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",확인 필요,#articleBody,1109,NaN
8,입법박람회에 등장한 사족로봇,확인 필요,#gisaview,2423,NaN
9,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",확인 필요,.article-txt-contents,832,NaN


전체 기사: 10 건
본문이 있는 기사: 10 건
본문이 없는 기사: 0 건


In [ ]:
from urllib.parse import urlparse

# 기사 주소에서 언론사 도메인 꺼내기
articles_df["domain"] = articles_df["original_url"].apply(
    lambda url: urlparse(url).netloc
)

# 언론사별로 사용된 본문 선택자 확인
selector_summary = (
    articles_df[["domain", "selector"]]
    .drop_duplicates()
    .sort_values("domain")
)

display(selector_summary)

,domain,selector
6,biz.chosun.com,.article-body
8,m.skyedaily.com,#gisaview
0,www.asiatime.co.kr,#articleContent
5,www.etnews.com,#articleBody
4,www.jejudomin.co.kr,#article-view-content-div
9,www.metroseoul.co.kr,.article-txt-contents
1,www.newsian.co.kr,#article-view-content-div
3,www.newsis.com,article
2,www.pinpointnews.co.kr,#article-view-content-div
7,www.techholic.co.kr,#articleBody


In [ ]:
# 스카이데일리는 PC 주소 변환이 필요하므로 우선 별도 처리
general_selectors = selector_summary[
    selector_summary["domain"] != "m.skyedaily.com"
]

# 언론사별로 확인된 선택자를 리스트로 정리
site_selectors = (
    general_selectors
    .groupby("domain")["selector"]
    .apply(list)
    .to_dict()
)

print("일반 본문 선택자")
for domain, selectors in site_selectors.items():
    print(f"{domain}: {selectors}")

print("\n별도 처리")
print("m.skyedaily.com: PC 주소 확인 및 인코딩 처리 필요")

일반 본문 선택자
biz.chosun.com: ['.article-body']
www.asiatime.co.kr: ['#articleContent']
www.etnews.com: ['#articleBody']
www.jejudomin.co.kr: ['#article-view-content-div']
www.metroseoul.co.kr: ['.article-txt-contents']
www.newsian.co.kr: ['#article-view-content-div']
www.newsis.com: ['article']
www.pinpointnews.co.kr: ['#article-view-content-div']
www.techholic.co.kr: ['#articleBody']

별도 처리
m.skyedaily.com: PC 주소 확인 및 인코딩 처리 필요


In [ ]:
from urllib.parse import urlparse


def get_article_markdown(url):
    """기사 주소 하나를 받아 본문을 마크다운으로 변환합니다."""

    result = {
        "original_url": url,
        "requested_url": url,
        "selector": None,
        "markdown": None,
        "status": "실패",
        "error": None,
    }

    try:
        domain = urlparse(url).netloc

        # 앞에서 확인한 스카이데일리 기사만 PC 주소로 변경
        is_sky_example = (
            domain == "m.skyedaily.com"
            and "ID=314172" in url
        )

        if is_sky_example:
            result["requested_url"] = (
                "https://www.skyedaily.com/news/news_view.html?ID=314172"
            )

        # 기사 페이지 요청
        response = requests.get(
            result["requested_url"],
            timeout=30,
        )
        response.raise_for_status()

        # 스카이데일리에서 확인한 인코딩 적용
        if is_sky_example:
            page_html = response.content.decode("cp949")
            soup = BeautifulSoup(page_html, "html.parser")

            selectors_to_try = ["#gisaview"]

        else:
            soup = BeautifulSoup(response.content, "html.parser")

            # 해당 언론사에서 확인한 선택자를 먼저 사용
            selectors_to_try = site_selectors.get(domain, [])

        # 언론사 규칙으로 찾지 못하면 공통 선택자도 시도
        selectors_to_try = selectors_to_try + article_selectors

        article_tag = None

        for selector in selectors_to_try:
            candidate = soup.select_one(selector)

            if candidate and candidate.get_text(" ", strip=True):
                article_tag = candidate
                result["selector"] = selector
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 스크립트와 스타일은 본문에서 제외
        for tag in article_tag.select("script, style"):
            tag.decompose()

        # 본문을 마크다운으로 변환
        result["markdown"] = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        )

        result["status"] = "확인 필요"

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except UnicodeDecodeError:
        result["error"] = "페이지의 문자 인코딩을 확인해야 합니다."

    return result

In [ ]:
# 기존에 성공했던 0번 기사로 함수 테스트
test_url = articles_df.loc[0, "original_url"]

test_result = get_article_markdown(test_url)

print("상태:", test_result["status"])
print("사용한 선택자:", test_result["selector"])
print("오류:", test_result["error"])

if test_result["markdown"]:
    display(Markdown(test_result["markdown"]))

상태: 확인 필요
사용한 선택자: #articleContent
오류: None


advertisement

advertisement

잔디광장 로봇 뛰는 ‘입법박람회’…시민 정책·신기술 한마당

본회의장 ‘조작기소 특검’ 충돌…野 전면 필리버스터 전운

국민의힘 “방탄·악법 박람회…국회 앞 맞불 여론전까지 검토”

[아시아타임즈=장유림 기자] 서울 여의도 국회의사당 잔디광장에서 미래 신기술과 시민 정책이 어우러진 ‘입법박람회’가 개막해 활기를 띠었다. 반면 본관 안 본회의장 앞은 ‘조작기소 특검법’ 상정을 둘러싼 여야의 극한 대치와 국민의힘의 무제한 토론(필리버스터) 예고로 무거운 전운이 감돌며 국회 안팎의 풍경은 극명한 대조를 이뤘다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500301.jpg)


5일 서울 여의도 국회의사당에서 ‘2026 국회 입법박람회’가열렸다.(사진=장유림 기자)

5일 오전 개막한 ‘2026 국회 입법박람회’는 국민의 아이디어를 정책과 법률로 잇겠다는 취지에 걸맞게 개막 첫날부터 시민들의 큰 호응을 얻었다. 개막식에서는 타악그룹 ‘타고’의 힘찬 모둠북 연주가 울려 퍼졌고, 어린이 2명이 낭독한 ‘국민희망입법제안문’을 현대자동차그룹의 4족 보행 로봇 ‘스팟(Spot)’이 조정식 국회의장에게 전달하는 이색 광경도 펼쳐졌다. 경제·문화·미래·입법 등 4개 권역으로 꾸려져 88개 기관·기업·단체가 참여한 잔디광장 부스에는 이른 시간부터 관람객의 발길이 이어졌다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500320.png)


5일 국회 중앙잔디광장에서 열린 2026 국회 입법박람회 경제존 안 아모레퍼시픽 부스에서 피부색 측정을 위해 시민들이 대기하고 있다.(사진=장유림기자)

특히 기업들이 실물 메모리 반도체와 휴머노이드 로봇 등을 선보인 경제존 부스는 시민들의 발걸음을 멈춰 세웠다. 4족 보행 로봇 주위로 시민들이 모여들었고 아모레퍼시픽 부스 앞에서는 최신 기술로 피부색을 측정하고 맞춤형 뷰티 솔루션을 체험하려는 관람객들로 긴 줄이 늘어섰다.

잔디광장 곳곳에서는 시민들이 자유롭게 부스를 오가며 체험에 참여했고, 아이들의 웃음소리가 끊이지 않았다. 행사에 참여한 이모(50) 씨는 “국회라고 하면 늘 엄숙하고 정쟁으로 싸우기만 하는 곳인 줄 알았는데, 바로 앞마당에서 활기찬 축제가 열리니 국회가 한결 친근하게 느껴진다”고 소감을 전했다.

advertisement

advertisement

미래존에서 정책 제안에 나선 이윤아(17·설월여고) 양도 “작년에 나갔던 정책 대회는 발표 시간이 30분밖에 안 돼 아쉬웠는데, 넓은 잔디광장에서 하루 종일 시민들에게 직접 정책을 설명하고 자유롭게 제안할 수 있어 뜻깊다”며 웃었다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500302.jpg)


국민의힘 장동혁 대표와 정점식 원내대표가 5일 국회에서 열린 의원총회에서 민주당 규탄 구호를 외치고 있다.(사진=연합뉴스)

같은 시각 국회 본관은 사정이 달랐다. 잔디광장의 ‘국민입법제안소’에는 민생을 개선해 달라는 시민들의 손글씨 제안서가 쌓여갔지만, 본관에서는 여야의 입법 충돌로 의사일정이 멈춰 섰다.

이날 오전 여야 원내대표는 윤석열 정부 당시 수사기관의 표적 수사 의혹을 파헤치는 특검법 상정을 두고 막판 담판을 벌였으나 합의점을 찾지 못했다. 더불어민주당(이하 민주당)이 특검법을 비롯해 민주유공자법, 초광역특구법 등 6개 안건을 이날 본회의에서 처리하겠다고 나서자 국민의힘은 상정되는 모든 법안에 필리버스터를 걸겠다며 맞섰다. 여야가 이견을 좁히지 못하면서 당초 오후 2시로 예정됐던 본회의는 오후 3시로 1시간 미뤄졌다.

국민의힘은 경내의 축제 분위기를 겨냥해 민주당을 비판했다. 장동혁 국민의힘 대표는 이날 오전 당 최고위원회의(이하 최고위)에서 “악법만 통과시키면서 입법박람회는 대체 왜 하는 것이냐”며 “입법박람회를 열어놓고 오늘 본회의에 조작기소 특검법을 상정한다면 국민들이 뭐라고 하겠느냐”고 비판했다. 이어 “박람회를 열고 이런 악법에 대해 야당에 필리버스터를 하라고 종용하는 꼴 아니냐”며 “강행한다면 박람회를 이재명 공소취소 박람회로 만드는 것”이라고 강조했다.

국민의힘 지도부는 장외 여론전까지 검토하고 있다. 최보윤 국민의힘 수석대변인은 이날 오전 최고위 직후 기자들과 만난 자리에서 “입법박람회가 진행 중인데, 국민들 보시기에 정말 ‘방탄 입법박람회’, ‘위헌 입법박람회’, ‘악법 입법박람회’라고 볼 수밖에 없다”고 꼬집었다. 이어 “입법박람회가 열리는 밖과, 국회 안에서 필리버스터를 진행할 수밖에 없는 굉장히 대비되는 상황”이라며 “국회 앞에서 별도로 목소리를 낼 수 있는 방안까지 검토하고 있다”고 밝혔다.

In [ ]:
# 테스트할 다른 아시아타임즈 기사 주소를 넣어주세요
test_url = "https://www.asiatime.co.kr/article/20261029500345"

test_result = get_article_markdown(test_url)

print("상태:", test_result["status"])
print("사용한 선택자:", test_result["selector"])
print("오류:", test_result["error"])

if test_result["markdown"]:
    display(Markdown(test_result["markdown"]))

상태: 확인 필요
사용한 선택자: #articleContent
오류: None


advertisement

advertisement

**인천 송도 바이오클러스터가 대규모 생산시설 투자를 앞세워 글로벌 바이오의약품 생산 거점으로 도약하고 있다. 삼성바이오로직스와 셀트리온, 롯데바이오로직스 등이 생산능력을 확대하며 외형을 키우고 있지만 원부자재 해외 의존과 공급망 불안 등 해결해야 할 과제도 남아 있다. 본지는 두 차례에 걸쳐 송도 바이오클러스터의 생산 경쟁력과 원부자재 자립 현황을 살펴보고 지속 가능한 성장 방안을 모색한다.**

삼성바이오·셀트리온·롯데바이오 공장 투자 진행

바이오의약품 수출 증가…국내 생산시설 확대 영향

빅파마, 생산 규모 포함 역량·경험 종합적 고려해 선정

[아시아타임즈=김지수 기자] 삼성바이오로직스와 셀트리온, 롯데바이오로직스 등 송도에 자리 잡은 국내 주요 바이오 기업들이 생산능력 확충에 속도를 내고 있다. 글로벌 바이오의약품 수요 확대에 대응해 생산시설을 늘리고 위탁개발생산(CDMO) 경쟁력을 강화하는 모습이다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/10/29/.cache/512/20261029500341.png)


삼성바이오로직스 4공장 전경. (사진=삼성바이오로직스)

2일 바이오업계에 따르면 삼성바이오로직스는 생산능력과 사업 포트폴리오, 사업 거점을 확대하는 '3대 축 확장' 전략을 추진하고 있다.

삼성바이오로직스는 지난해 4월 가동을 시작한 5공장(18만L)과 올해 3월 인수한 미국 록빌 생산시설(6만L)을 포함해 총 84만5000L의 생산능력을 확보했다.

송도 제2바이오캠퍼스에는 6~8공장 건설을 추진하고 있다. 이 중 6공장은 18만L 규모로 건설을 위한 이사회 최종 승인을 앞두고 있다.

advertisement

advertisement

계획대로 증설이 이뤄지면 2032년 국내 생산시설의 총생산능력은 132만5000L에 이른다. 미국 록빌 시설까지 포함하면 글로벌 생산능력은 138만5000L로 늘어난다.

삼성바이오로직스는 압도적인 생산 능력, 다각화된 사업 포트폴리오, 지리적 거점, 품질 경쟁력 등을 바탕으로 글로벌 고객의 신뢰를 높이고 지속적으로 수주를 확대해 나갈 계획이다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/10/29/.cache/512/20261029500343.jpg)


셀트리온 2공장 전경. (사진=셀트리온)

셀트리온은 기존 생산시설의 효율을 높이는 동시에 국내외 생산 거점을 확대하고 있다.

지난달 송도 원료의약품(DS)·완제의약품(DP) 생산시설 25만L에 대한 유럽의약품청(EMA)의 유럽 우수의약품 제조·품질관리 기준(EU-GMP) 인증을 갱신했다.

셀트리온은 이를 기반으로 올해 송도캠퍼스 전체 생산량을 전년 대비 약 15% 확대해 제품 판매 증가에 대응하고 생산 효율 개선을 통해 수익성 강화에 나선다는 방침이다.

이와 함께 1조2265억원을 투자해 총 18만L 규모의 4·5공장을 동시에 증설한다. 신설되는 4·5공장에는 최첨단 자동화 시스템과 스마트 팩토리 기술을 대거 적용해 생산 공정 효율과 유연성을 극대화할 예정이다.

미국 생산 거점도 확장한다. 당초 6만6000L로 계획했던 증설 규모를 7만5000L로 확대하면서 미국 공장의 원료의약품(DS) 생산능력은 기존 6만6000L에서 총 14만1000L로 늘어날 전망이다.

미국 현지 바이오의약품 생산에 대한 관심과 수요가 급증하면서 브랜치버그 생산시설은 셀트리온그룹의 현지 제품 공급과 CMO 사업 확대에도 중요한 역할을 맡을 전망이다.

롯데바이오로직스는 최근 전 직원의 송도 바이오 캠퍼스 입주를 마치고 본격적인 송도 시대를 열었다.

롯데바이오로직스는 송도 바이오캠퍼스 1공장의 주요 건설 공사를 마치고 사용 승인을 받았다. 해당 시설은 항체의약품 생산시설로 1만5000L 규모 스테인리스 스틸 바이오리액터 8기를 갖췄다. 총생산능력은 12만L다.

특히 미국 뉴욕 시러큐스 바이오 캠퍼스와 연계한 '듀얼 사이트(Dual Site)' 전략을 통해 초기 임상 생산부터 대규모 상업 생산까지 유기적으로 연결되는 통합 CDMO 서비스를 제공할 방침이다.

시러큐스 공장은 초기 개발 단계 프로젝트와 임상용 의약품 생산을 담당하고 송도 공장은 상업 생산을 맡는 방식이다.

송도 1공장의 상업 생산을 위한 준비도 진행 중이다. 롯데바이오로직스는 인천 송도 바이오캠퍼스 1공장의 의약품 제조 및 품질관리기준(GMP) 운영 준비를 추진하고 있다. 오는 2027년 상업 생산 개시를 목표로 하고 있다.

다만 생산능력 확대와 함께 글로벌 제약사의 수요에 맞춘 품질 관리와 안정적인 공급 역량을 확보하는 것도 중요하다.

바이오 업계에 따르면 글로벌 제약사들은 생산 규모뿐 아니라 품질 관리와 규제 대응 역량, 안정적인 공급 능력, 상업 생산 경험 등을 종합적으로 고려해 CDMO를 선정한다.

특히 기존 CDMO에서 다른 업체로 생산을 이전하려면 기술 이전과 공정 검증에 시간과 비용이 들기 때문에 신규 고객을 확보하기까지 진입 장벽이 높은 것으로 알려졌다.

바이오업계 관계자는 "빅파마는 단순히 생산을 위탁하는 것을 넘어 신약의 적기 출시와 안정적인 공급을 함께 책임질 파트너를 찾는다"며 "생산 규모 뿐만 아니라 품질 및 글로벌 규제 대응 역량, 납기 준수 능력, 상업 생산 경험, 고난도 신규 제형에 대한 기술력 등을 종합적으로 고려한다"고 말했다.

In [ ]:
import time

# 뉴스 검색 API 주소
api_url = "https://naverapihub.apigw.ntruss.com/search/v1/news"

# 받아온 기사 후보를 담을 리스트
all_news_items = []

# 1, 101, 201 ... 901번째 결과부터 각각 100건씩 요청
for start in range(1, 1000, 100):
    params = {
        "query": "아모레퍼시픽",
        "display": 100,
        "start": start,
        "sort": "date",  # 최신순
    }

    response = requests.get(
        api_url,
        headers=headers,
        params=params,
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    items = data.get("items", [])

    all_news_items.extend(items)

    print(
        f"{start}번째부터 {len(items)}건 조회 "
        f"· 누적 {len(all_news_items)}건"
    )

    # 더 이상 결과가 없으면 종료
    if len(items) < 100:
        break

    time.sleep(0.3)

print(f"\n총 받아온 검색 결과: {len(all_news_items)}건")

1번째부터 100건 조회 · 누적 100건
101번째부터 100건 조회 · 누적 200건
201번째부터 100건 조회 · 누적 300건
301번째부터 100건 조회 · 누적 400건
401번째부터 100건 조회 · 누적 500건
501번째부터 100건 조회 · 누적 600건
601번째부터 100건 조회 · 누적 700건
701번째부터 100건 조회 · 누적 800건
801번째부터 100건 조회 · 누적 900건
901번째부터 100건 조회 · 누적 1000건

총 받아온 검색 결과: 1000건


In [ ]:
# 검색 결과를 표로 정리
all_news_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in all_news_items
])

# 날짜를 한국 시간으로 변환
all_news_df["naver_provided_at"] = pd.to_datetime(
    all_news_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# 2023년 1월 1일 이후부터 현재까지
start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

filtered_news_df = all_news_df[
    all_news_df["naver_provided_at"].between(start_date, end_date)
].copy()

# 같은 원문 주소가 반복된 경우 하나만 남기기
filtered_news_df = filtered_news_df.drop_duplicates(
    subset=["original_url"]
)

print(f"기간과 중복을 정리한 기사 후보: {len(filtered_news_df)}건")
display(filtered_news_df.head(10))


기간과 중복을 정리한 기사 후보: 999건


,title,original_url,naver_url,naver_provided_at
0,"""국민 아이디어가 법·제도 된다""...'2026 입법 박람회' 5~6일 국회 개최",http://www.joseilbo.com/news/news_read.php?uid...,https://n.news.naver.com/mnews/article/123/000...,2026-11-05 15:22:00+09:00
1,"""신약 기업이 화장품 하면 밸류 깎인다?""…K-바이오, '바이오코스메틱'...",https://www.newsway.co.kr/news/view?ud=2026110...,https://www.newsway.co.kr/news/view?ud=2026110...,2026-11-05 15:22:00+09:00
2,"""신약 기술로 화장품 만든다""…제약·바이오, K뷰티 새 성장동력으로",https://www.news1.kr/industry/distribution/634...,https://n.news.naver.com/mnews/article/421/000...,2026-11-05 15:20:00+09:00
3,"[뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外",https://www.ibabynews.com/news/articleView.htm...,https://www.ibabynews.com/news/articleView.htm...,2026-11-05 15:18:00+09:00
4,로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막,http://www.issuenbiz.com/news/articleView.html...,http://www.issuenbiz.com/news/articleView.html...,2026-11-05 15:10:00+09:00
5,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,https://www.asiatime.co.kr/article/20261105500303,2026-11-05 15:06:00+09:00
6,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.htm...,https://www.newsian.co.kr/news/articleView.htm...,2026-11-05 14:54:00+09:00
7,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_00038...,https://n.news.naver.com/mnews/article/003/001...,2026-11-05 14:32:00+09:00
8,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleVie...,https://www.pinpointnews.co.kr/news/articleVie...,2026-11-05 14:32:00+09:00
9,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,https://www.jejudomin.co.kr/news/articleView.h...,2026-11-05 14:32:00+09:00


In [ ]:
# 먼저 기사 후보 20건만 선택
test_news_df = filtered_news_df.head(20)

# 원문 수집 결과를 담을 리스트
new_article_results = []

for index, row in test_news_df.iterrows():
    article_url = row["original_url"]

    # 원문 주소가 없는 경우 건너뛰기
    if not isinstance(article_url, str) or not article_url.strip():
        print(f"건너뜀: 원문 주소 없음 · {row['title']}")
        continue

    # 앞에서 만든 함수로 본문 가져오기
    result = get_article_markdown(article_url)

    # 검색 결과의 제목도 함께 기록
    result["title"] = row["title"]
    new_article_results.append(result)

    print(f"{result['status']} · {row['title']}")

    # 요청 사이에 잠시 대기
    time.sleep(1)

# 결과를 표로 변환
new_articles_df = pd.DataFrame(new_article_results)

print(f"\n확인한 기사: {len(new_articles_df)}건")
display(
    new_articles_df[["title", "status", "selector", "error"]]
)

실패 · "국민 아이디어가 법·제도 된다"...'2026 입법 박람회' 5~6일 국회 개최
확인 필요 · "신약 기업이 화장품 하면 밸류 깎인다?"…K-바이오, '바이오코스메틱'...
확인 필요 · "신약 기술로 화장품 만든다"…제약·바이오, K뷰티 새 성장동력으로
확인 필요 · [뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外
확인 필요 · 로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막
확인 필요 · [AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...
확인 필요 · 로봇·수소차·누리호 한자리에…국회 입법박람회 개막
확인 필요 · [현장]"신약 기술이 화장품으로" 바이오기업, 뷰티서 새 먹거리 찾는다
확인 필요 · 북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네
확인 필요 · 이니스프리 모음재단, 제주 '모드락마당' 조성
확인 필요 · “좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색
확인 필요 · [르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...
확인 필요 · 에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시
확인 필요 · 입법박람회에 등장한 사족로봇
확인 필요 · 에이피 뷰티, '하이엔드 미스트'..."피부과 시술 후 덧바르는 밴드"
확인 필요 · [포토] 2026 국회 입법박람회
확인 필요 · [K-소비재 3대축] GS25, 편의점 넘어 글로벌 플랫폼으로…K푸드 수출 키...
확인 필요 · [포토] 국회 잔디광장에 등장한 4족보행 로봇..."'국민입법제안서' 받으...
확인 필요 · 성수 뷰티 명소 '아모레 성수' 매장 이전, 인근 연무장길로 자리 옮긴다
확인 필요 · [포토] 맞춤형 화장품 제작 로봇

확인한 기사: 20건


,title,status,selector,error
0,"""국민 아이디어가 법·제도 된다""...'2026 입법 박람회' 5~6일 국회 개최",실패,NaN,본문 영역을 찾지 못했습니다.
1,"""신약 기업이 화장품 하면 밸류 깎인다?""…K-바이오, '바이오코스메틱'...",확인 필요,article,NaN
2,"""신약 기술로 화장품 만든다""…제약·바이오, K뷰티 새 성장동력으로",확인 필요,article,NaN
3,"[뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外",확인 필요,article,NaN
4,로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막,확인 필요,#article-view-content-div,NaN
5,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",확인 필요,#articleContent,NaN
6,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,확인 필요,#article-view-content-div,NaN
7,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",확인 필요,article,NaN
8,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,확인 필요,#article-view-content-div,NaN
9,"이니스프리 모음재단, 제주 '모드락마당' 조성",확인 필요,#article-view-content-div,NaN


In [ ]:
print(articles_df.loc[1, "original_url"])

https://www.newsian.co.kr/news/articleView.html?idxno=100699


In [ ]:
import hashlib
from urllib.parse import urlparse

# 뉴스 스키마로 변환한 결과를 담을 리스트
news_records = []

for _, row in articles_df.iterrows():
    article_url = row["original_url"]
    article_markdown = row["markdown"]

    # 원문 주소가 없는 기사는 건너뛰기
    if not isinstance(article_url, str) or not article_url.strip():
        print("원문 주소가 없어 건너뜀:", row["title"])
        continue

    # URL을 이용해 기사마다 고유한 문서 ID 생성
    url_hash = hashlib.sha256(
        article_url.encode("utf-8")
    ).hexdigest()[:16]

    # 기사 본문이 없는 경우 None으로 처리
    if not isinstance(article_markdown, str) or not article_markdown.strip():
        article_markdown = None

    # 스키마에 맞는 기사 한 건
    news_record = {
        "document_id": f"NEWS-{url_hash}",
        "title": row["title"],

        # 언론사 이름과 원문 발행일은 원문에서 확인 후 입력
        "source_name": None,
        "original_url": article_url,
        "published_at": None,

        # 실제 수집 시각을 기록하지 않았으므로 비워둠
        "collected_at": None,

        # 검색한 기업이지만 관련성은 아직 확인 전
        "company_relations": [
            {
                "company_id": "COMPANY-AMOREPACIFIC",
                "relevance": "판단 어려움",
                "review_status": "검토 필요",
            }
        ],

        # 주제 라벨은 이후 분류
        "topics": [],

        # 추출한 기사 본문
        "content": article_markdown,
        "content_type": "원문 추출" if article_markdown else None,

        # 기사별 이용 조건은 아직 확인 전
        "usage_status": "확인 필요",
    }

    news_records.append(news_record)

# 스키마 표로 변환
news_records_df = pd.DataFrame(news_records)

print(f"스키마에 담은 기사 후보: {len(news_records_df)}건")

display(
    news_records_df[
        [
            "document_id",
            "title",
            "original_url",
            "company_relations",
            "topics",
            "usage_status",
        ]
    ]
)

스키마에 담은 기사 후보: 10건


,document_id,title,original_url,company_relations,topics,usage_status
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.htm...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleVie...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_00038...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,https://www.etnews.com/20261105000343,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,https://biz.chosun.com/policy/politics/assembl...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",http://www.techholic.co.kr/news/articleView.ht...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/2026110550...,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[],확인 필요


In [ ]:
# content를 포함해서 표시
display(
    news_records_df[
        ["document_id", "title", "content"]
    ]
)

,document_id,title,content
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",advertisement\n\nadvertisement\n\n잔디광장 로봇 뛰는 ‘...
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,"［뉴시안= 이태영 기자］로봇과 수소차, 누리호 등 첨단산업 기술을 한자리에서 만나고..."
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,![네이버 증권.](https://cdn.pinpointnews.co.kr/news...
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다","## 제약사 연구기술부터 신약 후보물질까지 화장품 사업화 확대\n\n## 아모레 ""..."
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",![‘퐁낭정원’에 설치된 화장품 공병 업사이클링 스툴 [사진=이니스프리 모음재단]]...
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,[![한국바이오협회는 5일 서울 노보텔 앰배서더 동대문에서 '2026 바이오코스메틱...
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,국회가 국내 기업의 첨단 기술 전시장으로 탈바꿈했다. 국회는 5일 국회중앙잔디광장에...
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",[테크홀릭] 아모레퍼시픽은 하이엔드 스킨케어 브랜드 에이피 뷰티(AP BEAUTY)...
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,![](https://pds.skyedaily.com/news_data2025_2/...
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",![](//cdn.emetro.co.kr/data2/content/image/202...


In [ ]:
display(Markdown(news_records_df.loc[0, "content"]))

advertisement

advertisement

잔디광장 로봇 뛰는 ‘입법박람회’…시민 정책·신기술 한마당

본회의장 ‘조작기소 특검’ 충돌…野 전면 필리버스터 전운

국민의힘 “방탄·악법 박람회…국회 앞 맞불 여론전까지 검토”

[아시아타임즈=장유림 기자] 서울 여의도 국회의사당 잔디광장에서 미래 신기술과 시민 정책이 어우러진 ‘입법박람회’가 개막해 활기를 띠었다. 반면 본관 안 본회의장 앞은 ‘조작기소 특검법’ 상정을 둘러싼 여야의 극한 대치와 국민의힘의 무제한 토론(필리버스터) 예고로 무거운 전운이 감돌며 국회 안팎의 풍경은 극명한 대조를 이뤘다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500301.jpg)


5일 서울 여의도 국회의사당에서 ‘2026 국회 입법박람회’가열렸다.(사진=장유림 기자)

5일 오전 개막한 ‘2026 국회 입법박람회’는 국민의 아이디어를 정책과 법률로 잇겠다는 취지에 걸맞게 개막 첫날부터 시민들의 큰 호응을 얻었다. 개막식에서는 타악그룹 ‘타고’의 힘찬 모둠북 연주가 울려 퍼졌고, 어린이 2명이 낭독한 ‘국민희망입법제안문’을 현대자동차그룹의 4족 보행 로봇 ‘스팟(Spot)’이 조정식 국회의장에게 전달하는 이색 광경도 펼쳐졌다. 경제·문화·미래·입법 등 4개 권역으로 꾸려져 88개 기관·기업·단체가 참여한 잔디광장 부스에는 이른 시간부터 관람객의 발길이 이어졌다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500320.png)


5일 국회 중앙잔디광장에서 열린 2026 국회 입법박람회 경제존 안 아모레퍼시픽 부스에서 피부색 측정을 위해 시민들이 대기하고 있다.(사진=장유림기자)

특히 기업들이 실물 메모리 반도체와 휴머노이드 로봇 등을 선보인 경제존 부스는 시민들의 발걸음을 멈춰 세웠다. 4족 보행 로봇 주위로 시민들이 모여들었고 아모레퍼시픽 부스 앞에서는 최신 기술로 피부색을 측정하고 맞춤형 뷰티 솔루션을 체험하려는 관람객들로 긴 줄이 늘어섰다.

잔디광장 곳곳에서는 시민들이 자유롭게 부스를 오가며 체험에 참여했고, 아이들의 웃음소리가 끊이지 않았다. 행사에 참여한 이모(50) 씨는 “국회라고 하면 늘 엄숙하고 정쟁으로 싸우기만 하는 곳인 줄 알았는데, 바로 앞마당에서 활기찬 축제가 열리니 국회가 한결 친근하게 느껴진다”고 소감을 전했다.

advertisement

advertisement

미래존에서 정책 제안에 나선 이윤아(17·설월여고) 양도 “작년에 나갔던 정책 대회는 발표 시간이 30분밖에 안 돼 아쉬웠는데, 넓은 잔디광장에서 하루 종일 시민들에게 직접 정책을 설명하고 자유롭게 제안할 수 있어 뜻깊다”며 웃었다.

![image](//csossihettpx2597658.cdn.gov-ntruss.com/data2/content/image/2026/11/05/.cache/512/20261105500302.jpg)


국민의힘 장동혁 대표와 정점식 원내대표가 5일 국회에서 열린 의원총회에서 민주당 규탄 구호를 외치고 있다.(사진=연합뉴스)

같은 시각 국회 본관은 사정이 달랐다. 잔디광장의 ‘국민입법제안소’에는 민생을 개선해 달라는 시민들의 손글씨 제안서가 쌓여갔지만, 본관에서는 여야의 입법 충돌로 의사일정이 멈춰 섰다.

이날 오전 여야 원내대표는 윤석열 정부 당시 수사기관의 표적 수사 의혹을 파헤치는 특검법 상정을 두고 막판 담판을 벌였으나 합의점을 찾지 못했다. 더불어민주당(이하 민주당)이 특검법을 비롯해 민주유공자법, 초광역특구법 등 6개 안건을 이날 본회의에서 처리하겠다고 나서자 국민의힘은 상정되는 모든 법안에 필리버스터를 걸겠다며 맞섰다. 여야가 이견을 좁히지 못하면서 당초 오후 2시로 예정됐던 본회의는 오후 3시로 1시간 미뤄졌다.

국민의힘은 경내의 축제 분위기를 겨냥해 민주당을 비판했다. 장동혁 국민의힘 대표는 이날 오전 당 최고위원회의(이하 최고위)에서 “악법만 통과시키면서 입법박람회는 대체 왜 하는 것이냐”며 “입법박람회를 열어놓고 오늘 본회의에 조작기소 특검법을 상정한다면 국민들이 뭐라고 하겠느냐”고 비판했다. 이어 “박람회를 열고 이런 악법에 대해 야당에 필리버스터를 하라고 종용하는 꼴 아니냐”며 “강행한다면 박람회를 이재명 공소취소 박람회로 만드는 것”이라고 강조했다.

국민의힘 지도부는 장외 여론전까지 검토하고 있다. 최보윤 국민의힘 수석대변인은 이날 오전 최고위 직후 기자들과 만난 자리에서 “입법박람회가 진행 중인데, 국민들 보시기에 정말 ‘방탄 입법박람회’, ‘위헌 입법박람회’, ‘악법 입법박람회’라고 볼 수밖에 없다”고 꼬집었다. 이어 “입법박람회가 열리는 밖과, 국회 안에서 필리버스터를 진행할 수밖에 없는 굉장히 대비되는 상황”이라며 “국회 앞에서 별도로 목소리를 낼 수 있는 방안까지 검토하고 있다”고 밝혔다.

In [ ]:
import re

def remove_images(content):
    """마크다운과 HTML의 이미지 표시를 제거합니다."""

    if not isinstance(content, str):
        return content

    # 마크다운 이미지: ![설명](이미지주소)
    content = re.sub(
        r"!\[[^\]]*\]\((?:[^()]|\([^()]*\))*\)",
        "",
        content,
    )

    # HTML 이미지: <img ...>
    content = re.sub(
        r"<img\b[^>]*>",
        "",
        content,
        flags=re.IGNORECASE,
    )

    return content


# 뉴스 데이터의 content에서 이미지 제거
news_records_df["content"] = news_records_df["content"].apply(
    remove_images
)

print("기사 본문에서 이미지 표시를 제거했습니다.")

기사 본문에서 이미지 표시를 제거했습니다.


In [ ]:
# 전체 컬럼과 본문을 생략하지 않고 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(news_records_df)

document_id                                        title  \
0  NEWS-23178d84642bd6c9  [AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...   
1  NEWS-f1dc9d6e67264c8d                  로봇·수소차·누리호 한자리에…국회 입법박람회 개막   
2  NEWS-16bde7eaad888dd6           북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네   
3  NEWS-ab63f1790819521d      [현장]"신약 기술이 화장품으로" 바이오기업, 뷰티서 새 먹거리 찾는다   
4  NEWS-63d08bd21f6303a1                    이니스프리 모음재단, 제주 '모드락마당' 조성   
5  NEWS-e604b3c6f4d88af5          “좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색   
6  NEWS-db7f09305a3d8aef   [르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...   
7  NEWS-0fdd2f5d8f821b9a          에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시   
8  NEWS-c34adf013dd30022                              입법박람회에 등장한 사족로봇   
9  NEWS-e03b6b36b84b5050      에이피 뷰티, '하이엔드 미스트'..."피부과 시술 후 덧바르는 밴드"   

  source_name  \
0        None   
1        None   
2        None   
3        None   
4        None   
5        None   
6        None   
7        None   
8        None   
9        None   

                                                                                                                                   original_url  \
0                                                                                             https://www.asiatime.co.kr/article/20261105500303   
1                                                                                  https://www.newsian.co.kr/news/articleView.html?idxno=100699   
2                                                                             https://www.pinpointnews.co.kr/news/articleView.html?idxno=501603   
3                                                                                           https://www.newsis.com/view/NISX20261105_0003856394   
4                                                                                https://www.jejudomin.co.kr/news/articleView.html?idxno=402053   
5                                                                                                         https://www.etnews.com/20261105000343   
6  https://biz.chosun.com/policy/politics/assembly/2026/11/05/KXKBT46CRFBT5BYZTNJ7XQLQLE/?utm_source=naver&utm_medium=original&utm_campaign=biz   
7                                                                                 http://www.techholic.co.kr/news/articleView.html?idxno=223545   
8                                                                                              https://m.skyedaily.com/news_view.html?ID=314172   
9                                                                                            http://www.metroseoul.co.kr/article/20261105500277   

  published_at collected_at  \
0         None         None   
1         None         None   
2         None         None   
3         None         None   
4         None         None   
5         None         None   
6         None         None   
7         None         None   
8         None         None   
9         None         None   

                                                                           company_relations  \
0  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
1  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
2  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
3  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
4  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
5  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
6  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
7  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
8  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
9  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   

  to

In [ ]:
# 연도별 검색어
year_queries = [
    "아모레퍼시픽 2023",
    "아모레퍼시픽 2024",
    "아모레퍼시픽 2025",
    "아모레퍼시픽 2026",
]

# 연도별 검색 결과를 담을 리스트
year_news_items = []

for query in year_queries:
    print(f"\n검색 중: {query}")

    # 검색어마다 우선 최대 300건씩 가져오기
    for start in [1, 101, 201]:
        params = {
            "query": query,
            "display": 100,
            "start": start,
            "sort": "sim",  # 검색어 관련도순
        }

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        year_news_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        # 더 이상 검색 결과가 없으면 다음 검색어로 이동
        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n연도별 검색 결과: {len(year_news_items)}건")


검색 중: 아모레퍼시픽 2023
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 아모레퍼시픽 2024
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 아모레퍼시픽 2025
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 아모레퍼시픽 2026
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

연도별 검색 결과: 1200건


In [ ]:
# 연도별 검색 결과를 표로 정리
year_news_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_url": item.get("link"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in year_news_items
])

# 날짜를 한국 시간으로 변환
year_news_df["naver_provided_at"] = pd.to_datetime(
    year_news_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# 앞에서 수집한 최신 기사와 합치기
amore_candidates_df = pd.concat(
    [filtered_news_df, year_news_df],
    ignore_index=True,
)

# 2023년 1월 1일부터 현재까지 선택
start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

amore_candidates_df = amore_candidates_df[
    amore_candidates_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
].copy()

# 원문 주소가 없는 결과는 제외
amore_candidates_df = amore_candidates_df[
    amore_candidates_df["original_url"].notna()
    & amore_candidates_df["original_url"].str.strip().ne("")
]

# 같은 원문 주소는 하나만 남기기
amore_candidates_df = amore_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print(f"중복 제거 후 기사 후보: {len(amore_candidates_df)}건")
display(amore_candidates_df.head(10))

중복 제거 후 기사 후보: 1733건


,title,original_url,naver_url,naver_provided_at
0,"""국민 아이디어가 법·제도 된다""...'2026 입법 박람회' 5~6일 국회 개최",http://www.joseilbo.com/news/news_read.php?uid...,https://n.news.naver.com/mnews/article/123/000...,2026-11-05 15:22:00+09:00
1,"""신약 기업이 화장품 하면 밸류 깎인다?""…K-바이오, '바이오코스메틱'...",https://www.newsway.co.kr/news/view?ud=2026110...,https://www.newsway.co.kr/news/view?ud=2026110...,2026-11-05 15:22:00+09:00
2,"""신약 기술로 화장품 만든다""…제약·바이오, K뷰티 새 성장동력으로",https://www.news1.kr/industry/distribution/634...,https://n.news.naver.com/mnews/article/421/000...,2026-11-05 15:20:00+09:00
3,"[뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外",https://www.ibabynews.com/news/articleView.htm...,https://www.ibabynews.com/news/articleView.htm...,2026-11-05 15:18:00+09:00
4,로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막,http://www.issuenbiz.com/news/articleView.html...,http://www.issuenbiz.com/news/articleView.html...,2026-11-05 15:10:00+09:00
5,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,https://www.asiatime.co.kr/article/20261105500303,2026-11-05 15:06:00+09:00
6,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.htm...,https://www.newsian.co.kr/news/articleView.htm...,2026-11-05 14:54:00+09:00
7,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_00038...,https://n.news.naver.com/mnews/article/003/001...,2026-11-05 14:32:00+09:00
8,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleVie...,https://www.pinpointnews.co.kr/news/articleVie...,2026-11-05 14:32:00+09:00
9,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,https://www.jejudomin.co.kr/news/articleView.h...,2026-11-05 14:32:00+09:00


In [ ]:
year_counts = (
    amore_candidates_df["naver_provided_at"]
    .dt.year
    .value_counts()
    .sort_index()
)

print("연도별 기사 후보 수")
display(year_counts)

연도별 기사 후보 수


naver_provided_at
2024       9
2025     202
2026    1522
Name: count, dtype: int64

In [ ]:
# 네이버 검색 결과의 기사 주소와 제공 시각 가져오기
news_dates = news_df[
    ["original_url", "published_at"]
].drop_duplicates(subset=["original_url"])

# 기사 주소를 기준으로 날짜 연결
date_by_url = news_dates.set_index("original_url")["published_at"]

news_records_df["published_at"] = (
    news_records_df["original_url"]
    .map(date_by_url)
)

# 날짜를 한국 시간으로 변환
published_dates = pd.to_datetime(
    news_records_df["published_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

# JSON에 저장하기 쉬운 문자열로 변환
news_records_df["published_at"] = published_dates.apply(
    lambda value: value.isoformat() if pd.notna(value) else None
)

# 사용하지 않기로 한 컬럼 제거
news_records_df = news_records_df.drop(
    columns=[
        "source_name",
        "collected_at",
        "content_type",
        "usage_status",
    ],
    errors="ignore",
)

# 날짜가 들어갔는지 확인
display(
    news_records_df[["title", "published_at"]]
)

print(
    "날짜 확인이 필요한 기사:",
    news_records_df["published_at"].isna().sum(),
    "건",
)

,title,published_at
0,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",2026-11-05T15:06:00+09:00
1,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,2026-11-05T14:54:00+09:00
2,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,2026-11-05T14:32:00+09:00
3,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",2026-11-05T14:32:00+09:00
4,"이니스프리 모음재단, 제주 '모드락마당' 조성",2026-11-05T14:32:00+09:00
5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,2026-11-05T14:30:00+09:00
6,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,2026-11-05T14:18:00+09:00
7,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",2026-11-05T14:18:00+09:00
8,입법박람회에 등장한 사족로봇,2026-11-05T14:16:00+09:00
9,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",2026-11-05T14:14:00+09:00


날짜 확인이 필요한 기사: 0 건


In [ ]:
# 전체 컬럼과 본문 내용을 생략하지 않고 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(news_records_df)

print(f"전체 기사 수: {len(news_records_df)}건")
print(f"스키마 컬럼 수: {len(news_records_df.columns)}개")

document_id                                        title  \
0  NEWS-23178d84642bd6c9  [AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...   
1  NEWS-f1dc9d6e67264c8d                  로봇·수소차·누리호 한자리에…국회 입법박람회 개막   
2  NEWS-16bde7eaad888dd6           북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네   
3  NEWS-ab63f1790819521d      [현장]"신약 기술이 화장품으로" 바이오기업, 뷰티서 새 먹거리 찾는다   
4  NEWS-63d08bd21f6303a1                    이니스프리 모음재단, 제주 '모드락마당' 조성   
5  NEWS-e604b3c6f4d88af5          “좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색   
6  NEWS-db7f09305a3d8aef   [르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...   
7  NEWS-0fdd2f5d8f821b9a          에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시   
8  NEWS-c34adf013dd30022                              입법박람회에 등장한 사족로봇   
9  NEWS-e03b6b36b84b5050      에이피 뷰티, '하이엔드 미스트'..."피부과 시술 후 덧바르는 밴드"   

                                                                                                                                   original_url  \
0                                                                                             https://www.asiatime.co.kr/article/20261105500303   
1                                                                                  https://www.newsian.co.kr/news/articleView.html?idxno=100699   
2                                                                             https://www.pinpointnews.co.kr/news/articleView.html?idxno=501603   
3                                                                                           https://www.newsis.com/view/NISX20261105_0003856394   
4                                                                                https://www.jejudomin.co.kr/news/articleView.html?idxno=402053   
5                                                                                                         https://www.etnews.com/20261105000343   
6  https://biz.chosun.com/policy/politics/assembly/2026/11/05/KXKBT46CRFBT5BYZTNJ7XQLQLE/?utm_source=naver&utm_medium=original&utm_campaign=biz   
7                                                                                 http://www.techholic.co.kr/news/articleView.html?idxno=223545   
8                                                                                              https://m.skyedaily.com/news_view.html?ID=314172   
9                                                                                            http://www.metroseoul.co.kr/article/20261105500277   

                published_at  \
0  2026-11-05T15:06:00+09:00   
1  2026-11-05T14:54:00+09:00   
2  2026-11-05T14:32:00+09:00   
3  2026-11-05T14:32:00+09:00   
4  2026-11-05T14:32:00+09:00   
5  2026-11-05T14:30:00+09:00   
6  2026-11-05T14:18:00+09:00   
7  2026-11-05T14:18:00+09:00   
8  2026-11-05T14:16:00+09:00   
9  2026-11-05T14:14:00+09:00   

                                                                           company_relations  \
0  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
1  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
2  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
3  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
4  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
5  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
6  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
7  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
8  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   
9  [{'company_id': 'COMPANY-AMOREPACIFIC', 'relevance': '판단 어려움', 'review_status': '검토 필요'}]   

  topics  \
0     []   
1     []   
2     []   
3     []   
4     []   
5     []   
6     []   
7     []   
8     []   
9     []   

                                                

전체 기사 수: 10건
스키마 컬럼 수: 7개


In [ ]:
# 기사 단위 스키마에서 topics 제거
news_records_df = news_records_df.drop(
    columns=["topics"],
    errors="ignore",
)

# 변경된 스키마 확인
print("현재 스키마 컬럼:")
print(news_records_df.columns.tolist())

display(news_records_df)

현재 스키마 컬럼:
['document_id', 'title', 'original_url', 'published_at', 'company_relations', 'content']


,document_id,title,original_url,published_at,company_relations,content
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,2026-11-05T15:06:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",advertisement\n\nadvertisement\n\n잔디광장 로봇 뛰는 ‘...
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.htm...,2026-11-05T14:54:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","［뉴시안= 이태영 기자］로봇과 수소차, 누리호 등 첨단산업 기술을 한자리에서 만나고..."
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleVie...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","\n\n네이버 증권.\n\n중국 시장 의존도를 낮추고 북미와 유럽, 동남아시아 등으..."
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_00038...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","## 제약사 연구기술부터 신약 후보물질까지 화장품 사업화 확대\n\n## 아모레 ""..."
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",![‘퐁낭정원’에 설치된 화장품 공병 업사이클링 스툴 [사진=이니스프리 모음재단]]...
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,https://www.etnews.com/20261105000343,2026-11-05T14:30:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[](/tools/image_popup.html?v=bjQ5VkNYY2dLZTZMZ...
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,https://biz.chosun.com/policy/politics/assembl...,2026-11-05T14:18:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",국회가 국내 기업의 첨단 기술 전시장으로 탈바꿈했다. 국회는 5일 국회중앙잔디광장에...
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",http://www.techholic.co.kr/news/articleView.ht...,2026-11-05T14:18:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[테크홀릭] 아모레퍼시픽은 하이엔드 스킨케어 브랜드 에이피 뷰티(AP BEAUTY)...
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,2026-11-05T14:16:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",\n\n\n▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 ...
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/2026110550...,2026-11-05T14:14:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",\n\n\n에이피 뷰티 '듀얼 리페어 리프트 M.D. 크림 미스트' /아모레퍼시픽....


In [ ]:
import hashlib
import time

# 시도한 기사와 실패 기록을 보관할 변수
# 이미 만들어져 있다면 기존 기록을 유지합니다.
if "attempted_urls" not in globals():
    attempted_urls = set()

if "collection_errors" not in globals():
    collection_errors = []

# 이미 스키마에 들어간 기사 주소
existing_urls = set(
    news_records_df["original_url"].dropna()
)

# 아직 저장하지 않았고, 이번 작업에서 시도하지 않은 기사
remaining_news_df = amore_candidates_df[
    ~amore_candidates_df["original_url"].isin(
        existing_urls | attempted_urls
    )
].copy()

# 이번에 처리할 기사 수
batch_size = 20
batch_df = remaining_news_df.head(batch_size)

print(f"남은 기사 후보: {len(remaining_news_df)}건")
print(f"이번에 처리할 기사: {len(batch_df)}건")

남은 기사 후보: 1723건
이번에 처리할 기사: 20건


In [ ]:
# 이번에 성공한 기사를 담을 리스트
new_records = []

for number, (_, row) in enumerate(batch_df.iterrows(), start=1):
    article_url = row["original_url"]
    title = row["title"]

    # 같은 기사에 반복 요청하지 않도록 시도 기록
    attempted_urls.add(article_url)

    try:
        # 앞에서 만든 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)

        # 본문을 찾지 못하면 실패 목록에 남기고 다음 기사로 이동
        if not result.get("markdown"):
            collection_errors.append({
                "title": title,
                "original_url": article_url,
                "error": result.get("error"),
            })

            print(f"[{number}/{len(batch_df)}] 실패 · {title}")
            continue

        # 이미지 표시 제거
        content = remove_images(result["markdown"])

        # URL을 이용해 고유 문서 ID 생성
        url_hash = hashlib.sha256(
            article_url.encode("utf-8")
        ).hexdigest()[:16]

        # 네이버가 제공한 시각을 문자열로 변환
        provided_at = row["naver_provided_at"]
        published_at = (
            provided_at.isoformat()
            if pd.notna(provided_at)
            else None
        )

        # 현재 스키마에 맞게 기사 한 건 정리
        record = {
            "document_id": f"NEWS-{url_hash}",
            "title": title,
            "original_url": article_url,
            "published_at": published_at,
            "company_relations": [
                {
                    "company_id": "COMPANY-AMOREPACIFIC",
                    "relevance": "판단 어려움",
                    "review_status": "검토 필요",
                }
            ],
            "content": content,
        }

        new_records.append(record)
        print(f"[{number}/{len(batch_df)}] 본문 후보 확인 · {title}")

    except Exception as error:
        # 예상하지 못한 오류가 나더라도 다음 기사로 진행
        collection_errors.append({
            "title": title,
            "original_url": article_url,
            "error": f"{type(error).__name__}: {error}",
        })

        print(f"[{number}/{len(batch_df)}] 오류 · {title}")

    finally:
        # 언론사에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

# 기존 뉴스 표에 이번 결과 추가
if new_records:
    new_records_df = pd.DataFrame(new_records)

    news_records_df = pd.concat(
        [news_records_df, new_records_df],
        ignore_index=True,
    )

print("\n이번에 추가한 기사:", len(new_records), "건")
print("현재 스키마에 담긴 기사:", len(news_records_df), "건")
print("누적 실패 기사:", len(collection_errors), "건")

[1/20] 실패 · "국민 아이디어가 법·제도 된다"...'2026 입법 박람회' 5~6일 국회 개최
[2/20] 본문 후보 확인 · "신약 기업이 화장품 하면 밸류 깎인다?"…K-바이오, '바이오코스메틱'...
[3/20] 본문 후보 확인 · "신약 기술로 화장품 만든다"…제약·바이오, K뷰티 새 성장동력으로
[4/20] 본문 후보 확인 · [뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外
[5/20] 본문 후보 확인 · 로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막
[6/20] 본문 후보 확인 · [포토] 2026 국회 입법박람회
[7/20] 본문 후보 확인 · [K-소비재 3대축] GS25, 편의점 넘어 글로벌 플랫폼으로…K푸드 수출 키...
[8/20] 본문 후보 확인 · [포토] 국회 잔디광장에 등장한 4족보행 로봇..."'국민입법제안서' 받으...
[9/20] 본문 후보 확인 · 성수 뷰티 명소 '아모레 성수' 매장 이전, 인근 연무장길로 자리 옮긴다
[10/20] 본문 후보 확인 · [포토] 맞춤형 화장품 제작 로봇
[11/20] 본문 후보 확인 · [신상브리핑] 유니클로, 무신사 엠프티, 비욘드, 해태 빨라쪼, 오리온, ...
[12/20] 본문 후보 확인 · 에이피 뷰티, M.D. 라인에 크림 미스트 추가
[13/20] 본문 후보 확인 · 에이피 뷰티, 고기능성 크림 미스트 출시...M.D. 라인 신제품
[14/20] 실패 · 에이피 뷰티, PDRN 기반 크림 미스트 제품군 확대
[15/20] 본문 후보 확인 · "모공 1800분의 1 입자로 장벽 케어"…에이피 뷰티, '듀얼 리페어 리프트...
[16/20] 본문 후보 확인 · 입법박람회 '아모레퍼시픽' 부스 체험하는 조정식 국회의장
[17/20] 본문 후보 확인 · 퀸잇·SSF샵·SSG닷컴 등, 11월 쇼핑 축제 할인전 맞붙어
[18/20] 본문 후보 확인 · 아모레퍼시픽 부스 방문한 조정식 국회의장
[19/20] 본문 후보 확인 · 아모레퍼시픽 에이피 뷰

In [ ]:
# 정리 전 기사 수
before_count = len(news_records_df)

# 본문이 있는 기사만 남기기
news_records_df = news_records_df[
    news_records_df["content"].apply(
        lambda content: isinstance(content, str) and bool(content.strip())
    )
].reset_index(drop=True)

print(f"제외된 기사: {before_count - len(news_records_df)}건")
print(f"남은 기사: {len(news_records_df)}건")


제외된 기사: 0건
남은 기사: 28건


In [ ]:
news_records_df.head(28)

,document_id,title,original_url,published_at,company_relations,content
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,2026-11-05T15:06:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",advertisement\n\nadvertisement\n\n잔디광장 로봇 뛰는 ‘...
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.htm...,2026-11-05T14:54:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","［뉴시안= 이태영 기자］로봇과 수소차, 누리호 등 첨단산업 기술을 한자리에서 만나고..."
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleVie...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","\n\n네이버 증권.\n\n중국 시장 의존도를 낮추고 북미와 유럽, 동남아시아 등으..."
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_00038...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...","## 제약사 연구기술부터 신약 후보물질까지 화장품 사업화 확대\n\n## 아모레 ""..."
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,2026-11-05T14:32:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",![‘퐁낭정원’에 설치된 화장품 공병 업사이클링 스툴 [사진=이니스프리 모음재단]]...
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,https://www.etnews.com/20261105000343,2026-11-05T14:30:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[](/tools/image_popup.html?v=bjQ5VkNYY2dLZTZMZ...
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,https://biz.chosun.com/policy/politics/assembl...,2026-11-05T14:18:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",국회가 국내 기업의 첨단 기술 전시장으로 탈바꿈했다. 국회는 5일 국회중앙잔디광장에...
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",http://www.techholic.co.kr/news/articleView.ht...,2026-11-05T14:18:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",[테크홀릭] 아모레퍼시픽은 하이엔드 스킨케어 브랜드 에이피 뷰티(AP BEAUTY)...
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,2026-11-05T14:16:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",\n\n\n▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 ...
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/2026110550...,2026-11-05T14:14:00+09:00,"[{'company_id': 'COMPANY-AMOREPACIFIC', 'relev...",\n\n\n에이피 뷰티 '듀얼 리페어 리프트 M.D. 크림 미스트' /아모레퍼시픽....


In [ ]:
# company_relations 컬럼 제거
news_records_df = news_records_df.drop(
    columns=["company_relations"],
    errors="ignore",
)

# 모든 컬럼과 본문을 생략하지 않고 표로 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.max_rows", None,
):
    display(news_records_df)

print("현재 기사 수:", len(news_records_df), "건")
print("현재 스키마:", news_records_df.columns.tolist())

document_id                                           title  \
0   NEWS-23178d84642bd6c9     [AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...   
1   NEWS-f1dc9d6e67264c8d                     로봇·수소차·누리호 한자리에…국회 입법박람회 개막   
2   NEWS-16bde7eaad888dd6              북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네   
3   NEWS-ab63f1790819521d         [현장]"신약 기술이 화장품으로" 바이오기업, 뷰티서 새 먹거리 찾는다   
4   NEWS-63d08bd21f6303a1                       이니스프리 모음재단, 제주 '모드락마당' 조성   
5   NEWS-e604b3c6f4d88af5             “좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색   
6   NEWS-db7f09305a3d8aef      [르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...   
7   NEWS-0fdd2f5d8f821b9a             에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시   
8   NEWS-c34adf013dd30022                                 입법박람회에 등장한 사족로봇   
9   NEWS-e03b6b36b84b5050         에이피 뷰티, '하이엔드 미스트'..."피부과 시술 후 덧바르는 밴드"   
10  NEWS-37a6c221b0f79bc7     "신약 기업이 화장품 하면 밸류 깎인다?"…K-바이오, '바이오코스메틱'...   
11  NEWS-2ae270f156733027           "신약 기술로 화장품 만든다"…제약·바이오, K뷰티 새 성장동력으로   
12  NEWS-f4ac411a6eab26c3           [뷰티트렌드] 더샘, ‘먼작귀 에디션’ 출시…11월 올영픽 선정 外   
13  NEWS-f8a29ebb47d70d5f              로봇·수소차·누리호 한자리에…국회 '입법박람회' 5~6일 개막   
14  NEWS-ea4138d32a21d56a                              [포토] 2026 국회 입법박람회   
15  NEWS-594e008aa2fe8e81  [K-소비재 3대축] GS25, 편의점 넘어 글로벌 플랫폼으로…K푸드 수출 키...   
16  NEWS-1844fe9c06d5614e    [포토] 국회 잔디광장에 등장한 4족보행 로봇..."'국민입법제안서' 받으...   
17  NEWS-5decbb6a41e1167c        성수 뷰티 명소 '아모레 성수' 매장 이전, 인근 연무장길로 자리 옮긴다   
18  NEWS-b330b0fe90162f55                              [포토] 맞춤형 화장품 제작 로봇   
19  NEWS-95c452c170b545a1    [신상브리핑] 유니클로, 무신사 엠프티, 비욘드, 해태 빨라쪼, 오리온, ...   
20  NEWS-64d4ff01dce32358                      에이피 뷰티, M.D. 라인에 크림 미스트 추가   
21  NEWS-e2c6d7a044a9f574            에이피 뷰티, 고기능성 크림 미스트 출시...M.D. 라인 신제품   
22  NEWS-14e38cb40b24c1c2  "모공 1800분의 1 입자로 장벽 케어"…에이피 뷰티, '듀얼 리페어 리프트...   
23  NEWS-2521d2e4f8cba2cb                 입법박람회 '아모레퍼시픽' 부스 체험하는 조정식 국회의장   
24  NEWS-54a19c6a1edea584              퀸잇·SSF샵·SSG닷컴 등, 11월 쇼핑 축제 할인전 맞붙어   
25  NEWS-3199d0bd1e391a65                          아모레퍼시픽 부스 방문한 조정식 국회의장   
26  NEWS-7f357a40cb175452                아모레퍼시픽 에이피 뷰티, PDRN 함유 크림 미스트 출시   
27  NEWS-0d3b7b63da153b2f                      오리온 LG생활건강 해태제과 外(새로나왔어요!)   

                                                                                                                                    original_url  \
0                                                                                              https://www.asiatime.co.kr/article/20261105500303   
1                                                                                   https://www.newsian.co.kr/news/articleView.html?idxno=100699   
2                                                                              https://www.pinpointnews.co.kr/news/articleView.html?idxno=501603   
3                                                                                            https://www.newsis.com/view/NISX20261105_0003856394   
4                                                                                 https://www.jejudomin.co.kr/news/articleView.html?idxno=402053   
5                                                                                                          https://www.etnews.com/20261105000343   
6   https://biz.chosun.com/policy/politics/assembly/2026/11/05/KXKBT46CRFBT5BYZTNJ7XQLQLE/?utm_source=naver&utm_medium=original&utm_campaign=biz   
7                                                                                  http://www.techholic.co.kr/news/articleView.html?idxno=223545   
8                                                                                               https://m.skyedaily.com/news_view.html?ID=314172   
9                                                                                             http://www.metroseoul.co.kr/article/20261105500277   
10                                                                                    https://www.newsway.co.kr/news/view?ud=2026110515065064888   
11  

현재 기사 수: 28 건
현재 스키마: ['document_id', 'title', 'original_url', 'published_at', 'content']


In [ ]:
import hashlib
import time

# 기존 시도 기록이 없으면 새로 만들기
if "attempted_urls" not in globals():
    attempted_urls = set()

# 기존 뉴스 표의 인덱스 정리
news_records_df = news_records_df.reset_index(drop=True)

# 이미 담긴 기사 주소
existing_urls = set(
    news_records_df["original_url"].dropna()
)

# 원문 주소가 있는 후보만 선택
valid_candidates = amore_candidates_df[
    amore_candidates_df["original_url"].notna()
].copy()

valid_candidates = valid_candidates[
    valid_candidates["original_url"].str.strip().ne("")
]

# 이미 담았거나 시도한 기사는 제외
remaining_news_df = valid_candidates[
    ~valid_candidates["original_url"].isin(
        existing_urls | attempted_urls
    )
].copy()

success_count = 0
failure_count = 0

print(f"처리할 기사 후보: {len(remaining_news_df)}건")

for number, (_, row) in enumerate(remaining_news_df.iterrows(), start=1):
    article_url = row["original_url"]
    title = row["title"]

    # 이미 시도한 기사는 다시 요청하지 않도록 기록
    attempted_urls.add(article_url)

    try:
        # 앞에서 만든 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)
        article_markdown = result.get("markdown")

        # 본문을 가져오지 못하면 제외
        if not isinstance(article_markdown, str) or not article_markdown.strip():
            failure_count += 1
            continue

        # 이미지 제거
        content = remove_images(article_markdown)

        if not content.strip():
            failure_count += 1
            continue

        # 기사 주소로 고유 문서 ID 생성
        url_hash = hashlib.sha256(
            article_url.encode("utf-8")
        ).hexdigest()[:16]

        # 네이버 제공 시각
        provided_at = row["naver_provided_at"]
        published_at = (
            provided_at.isoformat()
            if pd.notna(provided_at)
            else None
        )

        # 현재 스키마에 맞게 저장
        news_records_df.loc[len(news_records_df)] = {
            "document_id": f"NEWS-{url_hash}",
            "title": title,
            "original_url": article_url,
            "published_at": published_at,
            "content": content,
        }

        success_count += 1

    except Exception:
        # 오류가 발생한 기사는 제외하고 다음 기사로 이동
        failure_count += 1

    finally:
        # 사이트에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

    # 20건마다 진행 상황 표시
    if number % 20 == 0:
        print(
            f"{number}/{len(remaining_news_df)}건 처리 · "
            f"추가 {success_count}건 · 제외 {failure_count}건"
        )

print("\n수집 완료")
print("이번에 추가한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("현재 전체 기사:", len(news_records_df), "건")

처리할 기사 후보: 1703건
20/1703건 처리 · 추가 16건 · 제외 4건
60/1703건 처리 · 추가 44건 · 제외 16건
80/1703건 처리 · 추가 62건 · 제외 18건
120/1703건 처리 · 추가 91건 · 제외 29건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


180/1703건 처리 · 추가 130건 · 제외 50건
220/1703건 처리 · 추가 159건 · 제외 61건
260/1703건 처리 · 추가 182건 · 제외 78건
280/1703건 처리 · 추가 198건 · 제외 82건
320/1703건 처리 · 추가 223건 · 제외 97건
360/1703건 처리 · 추가 251건 · 제외 109건
380/1703건 처리 · 추가 269건 · 제외 111건
400/1703건 처리 · 추가 284건 · 제외 116건
420/1703건 처리 · 추가 299건 · 제외 121건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


440/1703건 처리 · 추가 315건 · 제외 125건
460/1703건 처리 · 추가 328건 · 제외 132건
480/1703건 처리 · 추가 344건 · 제외 136건
500/1703건 처리 · 추가 355건 · 제외 145건
520/1703건 처리 · 추가 368건 · 제외 152건
540/1703건 처리 · 추가 383건 · 제외 157건
580/1703건 처리 · 추가 409건 · 제외 171건
600/1703건 처리 · 추가 427건 · 제외 173건
640/1703건 처리 · 추가 452건 · 제외 188건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


... (5 outputs left out)

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


1180/1703건 처리 · 추가 831건 · 제외 349건
1200/1703건 처리 · 추가 843건 · 제외 357건
1220/1703건 처리 · 추가 859건 · 제외 361건
1240/1703건 처리 · 추가 877건 · 제외 363건
1280/1703건 처리 · 추가 904건 · 제외 376건
1320/1703건 처리 · 추가 934건 · 제외 386건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


1340/1703건 처리 · 추가 947건 · 제외 393건
1360/1703건 처리 · 추가 963건 · 제외 397건
1380/1703건 처리 · 추가 976건 · 제외 404건
1400/1703건 처리 · 추가 991건 · 제외 409건
1440/1703건 처리 · 추가 1020건 · 제외 420건
1460/1703건 처리 · 추가 1038건 · 제외 422건
1480/1703건 처리 · 추가 1051건 · 제외 429건
1500/1703건 처리 · 추가 1064건 · 제외 436건
1520/1703건 처리 · 추가 1078건 · 제외 442건
1540/1703건 처리 · 추가 1094건 · 제외 446건
1580/1703건 처리 · 추가 1123건 · 제외 457건
1600/1703건 처리 · 추가 1139건 · 제외 461건
1620/1703건 처리 · 추가 1153건 · 제외 467건
1660/1703건 처리 · 추가 1180건 · 제외 480건
1680/1703건 처리 · 추가 1193건 · 제외 487건

수집 완료
이번에 추가한 기사: 1208 건
이번에 제외한 기사: 495 건
현재 전체 기사: 1236 건


In [ ]:
# 기사 수와 스키마 확인
print("전체 기사 수:", len(news_records_df), "건")
print("스키마:", news_records_df.columns.tolist())

# 앞에서 10건을 본문 생략 없이 표로 표시
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(news_records_df.head(10))

전체 기사 수: 1236 건
스키마: ['document_id', 'title', 'original_url', 'published_at', 'content']


document_id                                        title  \
0  NEWS-23178d84642bd6c9  [AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...   
1  NEWS-f1dc9d6e67264c8d                  로봇·수소차·누리호 한자리에…국회 입법박람회 개막   
2  NEWS-16bde7eaad888dd6           북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네   
3  NEWS-ab63f1790819521d      [현장]"신약 기술이 화장품으로" 바이오기업, 뷰티서 새 먹거리 찾는다   
4  NEWS-63d08bd21f6303a1                    이니스프리 모음재단, 제주 '모드락마당' 조성   
5  NEWS-e604b3c6f4d88af5          “좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색   
6  NEWS-db7f09305a3d8aef   [르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...   
7  NEWS-0fdd2f5d8f821b9a          에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시   
8  NEWS-c34adf013dd30022                              입법박람회에 등장한 사족로봇   
9  NEWS-e03b6b36b84b5050      에이피 뷰티, '하이엔드 미스트'..."피부과 시술 후 덧바르는 밴드"   

                                                                                                                                   original_url  \
0                                                                                             https://www.asiatime.co.kr/article/20261105500303   
1                                                                                  https://www.newsian.co.kr/news/articleView.html?idxno=100699   
2                                                                             https://www.pinpointnews.co.kr/news/articleView.html?idxno=501603   
3                                                                                           https://www.newsis.com/view/NISX20261105_0003856394   
4                                                                                https://www.jejudomin.co.kr/news/articleView.html?idxno=402053   
5                                                                                                         https://www.etnews.com/20261105000343   
6  https://biz.chosun.com/policy/politics/assembly/2026/11/05/KXKBT46CRFBT5BYZTNJ7XQLQLE/?utm_source=naver&utm_medium=original&utm_campaign=biz   
7                                                                                 http://www.techholic.co.kr/news/articleView.html?idxno=223545   
8                                                                                              https://m.skyedaily.com/news_view.html?ID=314172   
9                                                                                            http://www.metroseoul.co.kr/article/20261105500277   

                published_at  \
0  2026-11-05T15:06:00+09:00   
1  2026-11-05T14:54:00+09:00   
2  2026-11-05T14:32:00+09:00   
3  2026-11-05T14:32:00+09:00   
4  2026-11-05T14:32:00+09:00   
5  2026-11-05T14:30:00+09:00   
6  2026-11-05T14:18:00+09:00   
7  2026-11-05T14:18:00+09:00   
8  2026-11-05T14:16:00+09:00   
9  2026-11-05T14:14:00+09:00   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      

In [ ]:
# None, NaN 확인
missing_mask = news_records_df.isna()

# 빈 문자열이나 공백만 있는 값도 확인
blank_mask = news_records_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

# 두 조건 합치기
missing_mask = missing_mask | blank_mask

# 컬럼별 빈값 개수
missing_summary = pd.DataFrame({
    "빈값 수": missing_mask.sum(),
})

print(f"전체 기사: {len(news_records_df)}건")
display(missing_summary)

# 빈값이 하나라도 있는 기사 확인
missing_rows = news_records_df[missing_mask.any(axis=1)]

print(f"빈값이 있는 기사: {len(missing_rows)}건")
display(missing_rows)

전체 기사: 1236건


,빈값 수
document_id,0
title,0
original_url,0
published_at,0
content,0


빈값이 있는 기사: 0건


,document_id,title,original_url,published_at,content


In [ ]:
from pathlib import Path
import json
import os
import tempfile

# 저장 경로
output_path = Path(
    "/Users/anna/bulsae/data/collected/news.jsonl"
)

# 최종 스키마 컬럼
news_columns = [
    "document_id",
    "title",
    "original_url",
    "published_at",
    "content",
]

# 필요한 컬럼이 모두 있는지 확인
missing_columns = [
    column
    for column in news_columns
    if column not in news_records_df.columns
]

if missing_columns:
    raise ValueError(f"스키마에 없는 컬럼: {missing_columns}")

# 최종 컬럼만 선택
output_df = news_records_df[news_columns].copy()

if output_df.empty:
    raise ValueError("저장할 기사 데이터가 없습니다.")

# 빈값 확인
missing_values = output_df.isna() | output_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

if missing_values.any().any():
    raise ValueError("빈값이 있습니다. 저장 전에 확인해 주세요.")

# 문서 ID와 원문 주소 중복 확인
if output_df["document_id"].duplicated().any():
    raise ValueError("중복된 document_id가 있습니다.")

if output_df["original_url"].duplicated().any():
    raise ValueError("중복된 original_url이 있습니다.")

# JSONL 문자열로 변환
jsonl_text = output_df.to_json(
    orient="records",
    lines=True,
    force_ascii=False,
)

# 각 줄이 정상적인 JSON인지 확인
records = [
    json.loads(line)
    for line in jsonl_text.splitlines()
    if line.strip()
]

if len(records) != len(output_df):
    raise ValueError("저장할 데이터의 행 수가 일치하지 않습니다.")

# 임시 파일에 먼저 쓴 뒤 대상 파일 교체
output_path.parent.mkdir(parents=True, exist_ok=True)

with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=output_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)
    temp_file.write(jsonl_text)

try:
    os.replace(temp_path, output_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

print("저장 완료:", output_path)
print("저장된 기사:", len(records), "건")

저장 완료: /Users/anna/bulsae/data/collected/anna/news.jsonl
저장된 기사: 1236 건


In [ ]:
# 이니스프리 뉴스 검색
innisfree_params = {
    "query": "이니스프리",
    "display": 10,
    "start": 1,
    "sort": "date",
}

response = requests.get(
    "https://naverapihub.apigw.ntruss.com/search/v1/news",
    headers=headers,
    params=innisfree_params,
    timeout=30,
)
response.raise_for_status()

innisfree_data = response.json()

print("검색 결과 수:", innisfree_data.get("total"))
print("이번에 가져온 기사 수:", len(innisfree_data.get("items", [])))

검색 결과 수: 60562
이번에 가져온 기사 수: 10


In [ ]:
import time

# 이니스프리 검색어와 정렬 방식
innisfree_queries = [
    ("이니스프리", "date", 10),       # 최신순 최대 1,000건
    ("이니스프리 2023", "sim", 3),   # 관련도순 최대 300건
    ("이니스프리 2024", "sim", 3),
    ("이니스프리 2025", "sim", 3),
    ("이니스프리 2026", "sim", 3),
]

innisfree_items = []

for query, sort, max_pages in innisfree_queries:
    print(f"\n검색 중: {query}")

    for page in range(max_pages):
        start = 1 + page * 100

        params = {
            "query": query,
            "display": 100,
            "start": start,
            "sort": sort,
        }

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params=params,
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        innisfree_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n가져온 검색 결과: {len(innisfree_items)}건")


검색 중: 이니스프리
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건
  301번째부터 100건
  401번째부터 100건
  501번째부터 100건
  601번째부터 100건
  701번째부터 100건
  801번째부터 100건
  901번째부터 100건

검색 중: 이니스프리 2023
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2024
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2025
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2026
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

가져온 검색 결과: 2200건


In [ ]:
# 검색 결과를 표로 정리
innisfree_search_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in innisfree_items
])

# 네이버 제공 시각을 한국 시간으로 변환
innisfree_search_df["naver_provided_at"] = pd.to_datetime(
    innisfree_search_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

# 기간 안에 있고 원문 주소가 있는 기사만 선택
innisfree_candidates_df = innisfree_search_df[
    innisfree_search_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
    & innisfree_search_df["original_url"].notna()
    & innisfree_search_df["original_url"].str.strip().ne("")
].copy()

# 같은 원문 주소는 하나만 남기기
innisfree_candidates_df = innisfree_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print("중복 제거 후 이니스프리 기사 후보:", len(innisfree_candidates_df), "건")

# 연도별 후보 수 확인
year_counts = (
    innisfree_candidates_df["naver_provided_at"]
    .dt.year
    .value_counts()
    .sort_index()
)

display(year_counts)
display(innisfree_candidates_df.head(10))

중복 제거 후 이니스프리 기사 후보: 1787 건


naver_provided_at
2023     153
2024     230
2025     315
2026    1089
Name: count, dtype: int64

,title,original_url,naver_provided_at
0,"아모레퍼시픽 중국 C레벨 교체 이후 반등 신호, 김승환 설화수 흥행 기...",https://www.businesspost.co.kr/BP?command=arti...,2026-11-05 16:58:00+09:00
1,"APR·아모레, 해외 성장세 지속…LG생건, 해외 매출 감소",http://www.srtimes.kr/news/articleView.html?id...,2026-11-05 16:58:00+09:00
2,제주 고유의 자연과 문화를 담은 정원 '모드락마당',https://www.jejumaeil.net/news/articleView.htm...,2026-11-05 16:54:00+09:00
3,[2026 3Q 실적] K뷰티 실적 희비…아모레·LG생건·에이피알 '3색 전략',https://sports.hankooki.com/news/articleView.h...,2026-11-05 16:24:00+09:00
4,"이니스프리 모음재단·이지스자산운용, 제주에 ‘모드락마당’ 정원 조...",http://www.lak.co.kr/news/boardview.php?id=24262,2026-11-05 15:36:00+09:00
5,아모레 '성장' LG생건 '선방' 애경 '고전'…희비 엇갈린 뷰티 3대장,http://www.newstomato.com/ReadNews.aspx?no=131...,2026-11-05 15:32:00+09:00
6,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.h...,2026-11-05 14:32:00+09:00
7,"LG생활건강, 다마고치 협업 화장품 출시...IP 활용 확산",http://www.choicenews.co.kr/news/articleView.h...,2026-11-05 11:44:00+09:00
8,"11번가 '그랜드십일절', AI ·특가 효과 톡톡...흥행 열기 이어간다",https://www.mt.co.kr/living/2026/11/05/2026110...,2026-11-05 10:04:00+09:00
9,"11번가, 'AI 쇼핑' 그랜드십일절 흥행…초개인화 추천 거래액 3배↑",https://www.etnews.com/20261105000086,2026-11-05 09:53:00+09:00


In [10]:
import hashlib
import time

# 이니스프리 기사 결과가 아직 없으면 새로 만들기
if "innisfree_news_df" not in globals():
    innisfree_news_df = pd.DataFrame(
        columns=[
            "document_id",
            "title",
            "original_url",
            "published_at",
            "content",
        ]
    )

# 이미 시도한 기사 주소 기록
if "innisfree_attempted_urls" not in globals():
    innisfree_attempted_urls = set()

# 아직 시도하지 않은 기사만 선택
innisfree_remaining_df = innisfree_candidates_df[
    ~innisfree_candidates_df["original_url"].isin(
        innisfree_attempted_urls
    )
].copy()

success_count = 0
failure_count = 0

print(f"처리할 이니스프리 기사 후보: {len(innisfree_remaining_df)}건")

for number, (_, row) in enumerate(
    innisfree_remaining_df.iterrows(),
    start=1,
):
    article_url = row["original_url"]
    title = row["title"]

    # 이미 시도한 기사는 다시 요청하지 않도록 기록
    innisfree_attempted_urls.add(article_url)

    try:
        # 기존 함수로 기사 본문 가져오기
        result = get_article_markdown(article_url)
        article_markdown = result.get("markdown")

        if not isinstance(article_markdown, str) or not article_markdown.strip():
            # 본문을 가져오지 못한 기사는 제외
            failure_count += 1

        else:
            # 이미지 표시 제거
            content = remove_images(article_markdown)

            if not content.strip():
                failure_count += 1

            else:
                # URL을 이용해 문서 ID 생성
                url_hash = hashlib.sha256(
                    article_url.encode("utf-8")
                ).hexdigest()[:16]

                provided_at = row["naver_provided_at"]

                # 최종 뉴스 스키마에 맞게 정리
                record = {
                    "document_id": f"NEWS-{url_hash}",
                    "title": title,
                    "original_url": article_url,
                    "published_at": (
                        provided_at.isoformat()
                        if pd.notna(provided_at)
                        else None
                    ),
                    "content": content,
                }

                innisfree_news_df.loc[len(innisfree_news_df)] = record
                success_count += 1

    except Exception:
        # 오류가 난 기사는 제외하고 다음 기사로 이동
        failure_count += 1

    finally:
        # 사이트에 연속 요청을 보내지 않도록 대기
        time.sleep(1)

    # 20건마다 진행 상황 표시
    if number % 20 == 0:
        print(
            f"{number}/{len(innisfree_remaining_df)}건 처리 · "
            f"추가 {success_count}건 · 제외 {failure_count}건"
        )

print("\n수집 완료")
print("이번에 추가한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("이니스프리 전체 기사:", len(innisfree_news_df), "건")

NameError: name 'innisfree_candidates_df' is not defined

In [ ]:
# None, NaN 확인
missing_mask = innisfree_news_df.isna()

# 빈 문자열이나 공백만 있는 값 확인
blank_mask = innisfree_news_df.apply(
    lambda column: column.map(
        lambda value: isinstance(value, str) and not value.strip()
    )
)

# 두 조건 합치기
missing_mask = missing_mask | blank_mask

# 컬럼별 빈값 개수
missing_summary = pd.DataFrame({
    "빈값 수": missing_mask.sum(),
})

print("전체 기사:", len(innisfree_news_df), "건")
display(missing_summary)

# 빈값이 하나라도 있는 기사
missing_rows = innisfree_news_df[
    missing_mask.any(axis=1)
]

print("빈값이 있는 기사:", len(missing_rows), "건")
display(missing_rows)

전체 기사: 1227 건


,빈값 수
document_id,0
title,0
original_url,0
published_at,0
content,0


빈값이 있는 기사: 0 건


,document_id,title,original_url,published_at,content


In [ ]:
from pathlib import Path
import json
import os
import tempfile

# 기존 뉴스 파일
news_path = Path(
    "/Users/anna/bulsae/data/collected/news.jsonl"
)

# 기존 JSONL 읽기
with news_path.open(encoding="utf-8") as file:
    news_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

if not news_records:
    raise ValueError("뉴스 파일에 데이터가 없습니다.")

# 모든 기사에 아모레퍼시픽 기업 ID 추가
for record in news_records:
    record["company_ids"] = ["COMPANY-AMOREPACIFIC"]

# 임시 파일에 먼저 쓴 뒤 기존 파일 교체
with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=news_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)

    for record in news_records:
        temp_file.write(json.dumps(record, ensure_ascii=False) + "\n")

try:
    os.replace(temp_path, news_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

# 노트북의 아모레퍼시픽 표도 저장된 내용과 맞추기
news_records_df = pd.DataFrame(news_records)

print("업데이트 완료:", news_path)
print("아모레퍼시픽 ID가 추가된 기사:", len(news_records_df), "건")

display(
    news_records_df[["document_id", "title", "company_ids"]].head(10)
)

업데이트 완료: /Users/anna/bulsae/data/collected/anna/news.jsonl
아모레퍼시픽 ID가 추가된 기사: 1236 건


,document_id,title,company_ids
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",[COMPANY-AMOREPACIFIC]
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,[COMPANY-AMOREPACIFIC]
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,[COMPANY-AMOREPACIFIC]
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",[COMPANY-AMOREPACIFIC]
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",[COMPANY-AMOREPACIFIC]
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,[COMPANY-AMOREPACIFIC]
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,[COMPANY-AMOREPACIFIC]
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",[COMPANY-AMOREPACIFIC]
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,[COMPANY-AMOREPACIFIC]
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",[COMPANY-AMOREPACIFIC]


In [8]:
import pandas as pd

# 저장된 뉴스 JSONL 불러오기
news_path = "/Users/anna/bulsae/data/collected/news.jsonl"

news_records_df = pd.read_json(
    news_path,
    lines=True,
    convert_dates=False,
)

print("불러온 기사 수:", len(news_records_df), "건")
print("컬럼:", news_records_df.columns.tolist())

# 기사 10건 확인
with pd.option_context(
    "display.max_columns", None,
    "display.max_colwidth", 100,
):
    display(news_records_df.head(10))

불러온 기사 수: 1236 건
컬럼: ['document_id', 'title', 'original_url', 'published_at', 'content', 'company_ids']


/var/folders/ly/wq_1wj811fd359fys46rpjhm0000gn/T/ipykernel_24676/3914450181.py:6: Pandas4Warning: The 'convert_dates' keyword in read_json is deprecated and will be removed in a future version. Pass dtype=False to disable type conversion, or parse date columns with pd.to_datetime after reading.
  news_records_df = pd.read_json(


,document_id,title,original_url,published_at,content,company_ids
0,NEWS-23178d84642bd6c9,"[AT 현장] 잔디밭선 ‘축제’, 본회의선 ‘필버 전쟁’…국회 안팎 ‘극...",https://www.asiatime.co.kr/article/20261105500303,2026-11-05T15:06:00+09:00,advertisement\n\nadvertisement\n\n잔디광장 로봇 뛰는 ‘입법박람회’…시민 정책·신기술 한마당\n\n본회의장 ‘조작기소 특검’ 충돌…野 전면 필리버...,[COMPANY-AMOREPACIFIC]
1,NEWS-f1dc9d6e67264c8d,로봇·수소차·누리호 한자리에…국회 입법박람회 개막,https://www.newsian.co.kr/news/articleView.html?idxno=100699,2026-11-05T14:54:00+09:00,"［뉴시안= 이태영 기자］로봇과 수소차, 누리호 등 첨단산업 기술을 한자리에서 만나고 국민이 직접 입법 아이디어를 제안할 수 있는 국회 입법박람회가 막을 올렸다.\n\n![조...",[COMPANY-AMOREPACIFIC]
2,NEWS-16bde7eaad888dd6,북미·유럽으로 뻗는 K-뷰티…펌텍코리아·한국콜마 주가 화색도네,https://www.pinpointnews.co.kr/news/articleView.html?idxno=501603,2026-11-05T14:32:00+09:00,"\n\n네이버 증권.\n\n중국 시장 의존도를 낮추고 북미와 유럽, 동남아시아 등으로 해외 사업 영역을 확대하는 화장품 관련주가 5일 주목받고 있다.\n\n글로벌 시장에서...",[COMPANY-AMOREPACIFIC]
3,NEWS-ab63f1790819521d,"[현장]""신약 기술이 화장품으로"" 바이오기업, 뷰티서 새 먹거리 찾는다",https://www.newsis.com/view/NISX20261105_0003856394,2026-11-05T14:32:00+09:00,"## 제약사 연구기술부터 신약 후보물질까지 화장품 사업화 확대\n\n## 아모레 ""과학적 검증 필수""·투자업계 ""시장성·유통 역량 중요""\n\n| [서울=뉴시스] 권민지 기...",[COMPANY-AMOREPACIFIC]
4,NEWS-63d08bd21f6303a1,"이니스프리 모음재단, 제주 '모드락마당' 조성",https://www.jejudomin.co.kr/news/articleView.html?idxno=402053,2026-11-05T14:32:00+09:00,![‘퐁낭정원’에 설치된 화장품 공병 업사이클링 스툴 [사진=이니스프리 모음재단]](https://cdn.jejudomin.co.kr/news/photo/202611/402...,[COMPANY-AMOREPACIFIC]
5,NEWS-e604b3c6f4d88af5,“좋은 기술도 팔려야 혁신”…바이오·화장품업계 사업화 협력 모색,https://www.etnews.com/20261105000343,2026-11-05T14:30:00+09:00,[](/tools/image_popup.html?v=bjQ5VkNYY2dLZTZMZFBUbmNzandvWmpGcjBTK1pHYm1CUVRXR0pTMldlYVZkdE5JeGJ...,[COMPANY-AMOREPACIFIC]
6,NEWS-db7f09305a3d8aef,[르포] ‘4족 보행 로봇’이 국회의장에 입법 제안문 전달…“현장의 아...,https://biz.chosun.com/policy/politics/assembly/2026/11/05/KXKBT46CRFBT5BYZTNJ7XQLQLE/?utm_sourc...,2026-11-05T14:18:00+09:00,국회가 국내 기업의 첨단 기술 전시장으로 탈바꿈했다. 국회는 5일 국회중앙잔디광장에서 '입법박람회'를 개최했다. 지방자치단체나 정부기관이 중심이 됐던 과거 입법박람회와 달리...,[COMPANY-AMOREPACIFIC]
7,NEWS-0fdd2f5d8f821b9a,"에이피 뷰티, '듀얼 리페어 리프트 M.D. 크림 미스트' 출시",http://www.techholic.co.kr/news/articleView.html?idxno=223545,2026-11-05T14:18:00+09:00,[테크홀릭] 아모레퍼시픽은 하이엔드 스킨케어 브랜드 에이피 뷰티(AP BEAUTY)가 피부 특수 케어 솔루션인 M.D. 라인의 신제품 ‘듀얼 리페어 리프트 M.D. 크림 미...,[COMPANY-AMOREPACIFIC]
8,NEWS-c34adf013dd30022,입법박람회에 등장한 사족로봇,https://m.skyedaily.com/news_view.html?ID=314172,2026-11-05T14:16:00+09:00,\n\n\n▲ 조정식 국회의장이 5일 서울 여의도 국회 잔디광장에서 열린 2026 국회 입법박람회에서 사족보행 로봇으로부터 국민희망입법제안문을 받고 있다. 이번 행사는 국민...,[COMPANY-AMOREPACIFIC]
9,NEWS-e03b6b36b84b5050,"에이피 뷰티, '하이엔드 미스트'...""피부과 시술 후 덧바르는 밴드""",http://www.metroseoul.co.kr/article/20261105500277,2026-11-05T14:14:00+09:00,\n\n\n에이피 뷰티 '듀얼 리페어 리프트 M.D. 크림 미스트' /아모레퍼시픽.\n\n아모레퍼시픽의 하이엔드 스킨케어 브랜드 에이피 뷰티가 스킨케어 핵심 성분을 미스트 ...,[COMPANY-AMOREPACIFIC]


In [9]:
from pathlib import Path
import json
import os
import tempfile

# 이니스프리 수집 결과가 메모리에 있는지 확인
if "innisfree_news_df" not in globals():
    raise ValueError(
        "이니스프리 수집 결과가 메모리에 없습니다. "
        "별도로 저장해 둔 파일이 있는지 먼저 확인해 주세요."
    )

if innisfree_news_df.empty:
    raise ValueError("이니스프리 기사 데이터가 비어 있습니다.")

news_path = Path(
    "/Users/anna/bulsae/data/collected/news.jsonl"
)

# 기존 아모레퍼시픽 기사 읽기
with news_path.open(encoding="utf-8") as file:
    existing_records = [
        json.loads(line)
        for line in file
        if line.strip()
    ]

# 원문 주소로 기존 기사 찾기
records_by_url = {
    record["original_url"]: record
    for record in existing_records
}

# 이니스프리 표를 JSON 객체 목록으로 변환
innisfree_records = json.loads(
    innisfree_news_df.to_json(
        orient="records",
        date_format="iso",
        force_ascii=False,
    )
)

added_count = 0
linked_count = 0

for record in innisfree_records:
    article_url = record.get("original_url")

    if not article_url:
        continue

    if article_url in records_by_url:
        # 이미 있는 기사라면 기업 ID만 추가
        existing = records_by_url[article_url]
        company_ids = existing.get("company_ids") or []

        if "COMPANY-INNISFREE" not in company_ids:
            company_ids.append("COMPANY-INNISFREE")
            existing["company_ids"] = company_ids
            linked_count += 1

    else:
        # 새로운 기사라면 이니스프리 ID를 넣어 추가
        record["company_ids"] = ["COMPANY-INNISFREE"]
        existing_records.append(record)
        records_by_url[article_url] = record
        added_count += 1

# 임시 파일에 쓴 뒤 기존 JSONL 교체
with tempfile.NamedTemporaryFile(
    mode="w",
    encoding="utf-8",
    dir=news_path.parent,
    delete=False,
) as temp_file:
    temp_path = Path(temp_file.name)

    for record in existing_records:
        temp_file.write(json.dumps(record, ensure_ascii=False) + "\n")

try:
    os.replace(temp_path, news_path)
finally:
    if temp_path.exists():
        temp_path.unlink()

# 노트북 표도 저장된 데이터와 맞추기
news_records_df = pd.DataFrame(existing_records)

print("새로 추가한 이니스프리 기사:", added_count, "건")
print("기존 기사에 이니스프리 ID를 추가한 건수:", linked_count, "건")
print("저장된 전체 기사:", len(news_records_df), "건")
print("저장 위치:", news_path)

ValueError: 이니스프리 수집 결과가 메모리에 없습니다. 별도로 저장해 둔 파일이 있는지 먼저 확인해 주세요.

In [11]:
import os
import re
import html
import time

from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

# API 키 불러오기
load_dotenv("/Users/anna/bulsae/.env", override=True)

naver_client_id = os.getenv("NAVER_CLIENT_ID")
naver_client_secret = os.getenv("NAVER_CLIENT_SECRET")

if not naver_client_id or not naver_client_secret:
    raise ValueError(".env에서 네이버 API 키를 찾을 수 없습니다.")

headers = {
    "X-NCP-APIGW-API-KEY-ID": naver_client_id,
    "X-NCP-APIGW-API-KEY": naver_client_secret,
}

print("네이버 API 준비 완료")

네이버 API 준비 완료


In [12]:
def clean_news_text(text):
    """검색 결과의 HTML 태그와 특수문자 정리"""
    text = re.sub(r"<[^>]+>", "", text or "")
    return html.unescape(text).strip()


# 최신순 검색과 연도별 검색
current_year = pd.Timestamp.now(tz="Asia/Seoul").year

innisfree_queries = [("이니스프리", "date", 10)]

for year in range(2023, current_year + 1):
    innisfree_queries.append(
        (f"이니스프리 {year}", "sim", 3)
    )

innisfree_items = []

for query, sort, max_pages in innisfree_queries:
    print(f"\n검색 중: {query}")

    for page in range(max_pages):
        start = 1 + page * 100

        response = requests.get(
            "https://naverapihub.apigw.ntruss.com/search/v1/news",
            headers=headers,
            params={
                "query": query,
                "display": 100,
                "start": start,
                "sort": sort,
            },
            timeout=30,
        )
        response.raise_for_status()

        items = response.json().get("items", [])
        innisfree_items.extend(items)

        print(f"  {start}번째부터 {len(items)}건")

        if len(items) < 100:
            break

        time.sleep(0.3)

print(f"\n가져온 검색 결과: {len(innisfree_items)}건")


검색 중: 이니스프리
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건
  301번째부터 100건
  401번째부터 100건
  501번째부터 100건
  601번째부터 100건
  701번째부터 100건
  801번째부터 100건
  901번째부터 100건

검색 중: 이니스프리 2023
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2024
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2025
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

검색 중: 이니스프리 2026
  1번째부터 100건
  101번째부터 100건
  201번째부터 100건

가져온 검색 결과: 2200건


In [13]:
# 검색 결과를 표로 변환
innisfree_candidates_df = pd.DataFrame([
    {
        "title": clean_news_text(item.get("title")),
        "original_url": item.get("originallink"),
        "naver_provided_at": item.get("pubDate"),
    }
    for item in innisfree_items
])

# 네이버 제공 시각을 한국 시간으로 변환
innisfree_candidates_df["naver_provided_at"] = pd.to_datetime(
    innisfree_candidates_df["naver_provided_at"],
    errors="coerce",
    utc=True,
).dt.tz_convert("Asia/Seoul")

start_date = pd.Timestamp("2023-01-01", tz="Asia/Seoul")
end_date = pd.Timestamp.now(tz="Asia/Seoul")

# 기간 안에 있고 원문 주소가 있는 기사만 남기기
innisfree_candidates_df = innisfree_candidates_df[
    innisfree_candidates_df["naver_provided_at"].between(
        start_date,
        end_date,
    )
    & innisfree_candidates_df["original_url"].notna()
    & innisfree_candidates_df["original_url"].str.strip().ne("")
].copy()

# 같은 원문 주소 제거
innisfree_candidates_df = innisfree_candidates_df.drop_duplicates(
    subset=["original_url"]
).reset_index(drop=True)

print("중복 제거 후 기사 후보:", len(innisfree_candidates_df), "건")

# 후보 목록을 별도 파일로 저장
candidates_path = Path(
    "/Users/anna/bulsae/private_data/innisfree/innisfree_candidates.jsonl"
)

if candidates_path.exists():
    raise FileExistsError(
        f"이미 후보 파일이 있습니다. 덮어쓰지 않았습니다: {candidates_path}"
    )

with candidates_path.open("x", encoding="utf-8") as file:
    file.write(
        innisfree_candidates_df.to_json(
            orient="records",
            lines=True,
            force_ascii=False,
            date_format="iso",
        )
    )

print("후보 목록 저장 완료:", candidates_path)

중복 제거 후 기사 후보: 1787 건
후보 목록 저장 완료: /Users/anna/bulsae/private_data/innisfree_candidates.jsonl


In [14]:
import hashlib
import json
import time

from bs4 import BeautifulSoup
from markdownify import markdownify
from urllib.parse import urlparse


# 앞에서 확인한 언론사별 본문 위치
site_selectors = {
    "www.asiatime.co.kr": ["#articleContent"],
    "www.metroseoul.co.kr": [".article-txt-contents"],
    "biz.chosun.com": [".article-body"],
    "www.etnews.com": ["#articleBody"],
    "www.techholic.co.kr": ["#articleBody"],
    "www.jejudomin.co.kr": ["#article-view-content-div"],
    "www.newsian.co.kr": ["#article-view-content-div"],
    "www.pinpointnews.co.kr": ["#article-view-content-div"],
    "www.newsis.com": ["article"],
}

# 사이트별 규칙으로 찾지 못했을 때 시도할 위치
common_selectors = [
    "#article-view-content-div",
    "#articleContent",
    ".article-txt-contents",
    "#articleBody",
    "#article-body",
    "#newsContent",
    ".article-body",
    ".article-content",
    ".news_view",
    "article",
]


def get_article_markdown(url):
    """기사 주소에서 본문 후보를 찾아 마크다운으로 변환합니다."""

    result = {
        "markdown": None,
        "error": None,
    }

    try:
        domain = urlparse(url).netloc.lower()
        request_url = url

        # 이전에 확인한 스카이데일리 기사에만 적용
        is_sky_example = (
            domain == "m.skyedaily.com"
            and "ID=314172" in url
        )

        if is_sky_example:
            request_url = (
                "https://www.skyedaily.com/news/news_view.html?ID=314172"
            )

        response = requests.get(request_url, timeout=30)
        response.raise_for_status()

        # 해당 스카이데일리 기사에서는 cp949 사용
        if is_sky_example:
            page_html = response.content.decode("cp949")
            soup = BeautifulSoup(page_html, "html.parser")
            selectors = ["#gisaview"]
        else:
            soup = BeautifulSoup(response.content, "html.parser")
            selectors = site_selectors.get(domain, [])

        # 공통 선택자도 순서대로 시도
        selectors = selectors + common_selectors

        article_tag = None

        for selector in dict.fromkeys(selectors):
            candidate = soup.select_one(selector)

            if candidate and candidate.get_text(" ", strip=True):
                article_tag = candidate
                break

        if article_tag is None:
            result["error"] = "본문 영역을 찾지 못했습니다."
            return result

        # 이미지와 스크립트 제거
        for tag in article_tag.select("img, script, style, iframe"):
            tag.decompose()

        content = markdownify(
            str(article_tag),
            heading_style="ATX",
            table_infer_header=True,
        ).strip()

        if not content:
            result["error"] = "이미지 제거 후 본문이 비어 있습니다."
            return result

        result["markdown"] = content

    except requests.RequestException as error:
        result["error"] = f"페이지 요청 실패: {type(error).__name__}"

    except UnicodeDecodeError:
        result["error"] = "문자 인코딩 확인 필요"

    except Exception as error:
        result["error"] = f"변환 실패: {type(error).__name__}"

    return result

In [15]:
# 앞에서 저장한 기사 후보 파일
candidates_path = Path(
    "/Users/anna/bulsae/private_data/innisfree/innisfree_candidates.jsonl"
)

# 성공한 기사를 중간 저장할 파일
checkpoint_path = Path(
    "/Users/anna/bulsae/private_data/innisfree/innisfree_news_checkpoint.jsonl"
)

# 실패한 기사 주소와 이유를 기록할 파일
errors_path = Path(
    "/Users/anna/bulsae/private_data/innisfree/innisfree_news_errors.jsonl"
)


def read_jsonl(path):
    """JSONL 파일이 있으면 읽고, 없으면 빈 리스트를 반환합니다."""
    if not path.exists():
        return []

    with path.open(encoding="utf-8") as file:
        return [
            json.loads(line)
            for line in file
            if line.strip()
        ]


def append_jsonl(path, record):
    """JSON 객체 한 건을 파일에 추가합니다."""
    with path.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


# 후보, 기존 성공 결과, 기존 실패 기록 읽기
candidates = read_jsonl(candidates_path)
saved_articles = read_jsonl(checkpoint_path)
failed_articles = read_jsonl(errors_path)

# 이미 성공했거나 실패한 기사는 다시 요청하지 않음
processed_urls = {
    record["original_url"]
    for record in saved_articles + failed_articles
}

remaining = [
    record
    for record in candidates
    if record["original_url"] not in processed_urls
]

print("전체 후보:", len(candidates), "건")
print("이전에 저장한 기사:", len(saved_articles), "건")
print("이전에 제외한 기사:", len(failed_articles), "건")
print("이번에 처리할 기사:", len(remaining), "건")

success_count = 0
failure_count = 0

for number, candidate in enumerate(remaining, start=1):
    article_url = candidate["original_url"]
    title = candidate["title"]

    try:
        result = get_article_markdown(article_url)

        if result["markdown"]:
            # 기사 주소를 이용해 고유 ID 생성
            url_hash = hashlib.sha256(
                article_url.encode("utf-8")
            ).hexdigest()[:16]

            # 최종 뉴스 스키마
            record = {
                "document_id": f"NEWS-{url_hash}",
                "title": title,
                "original_url": article_url,
                "published_at": candidate["naver_provided_at"],
                "content": result["markdown"],
                "company_ids": ["COMPANY-INNISFREE"],
            }

            # 성공한 기사 한 건을 바로 저장
            append_jsonl(checkpoint_path, record)
            success_count += 1

        else:
            # 본문을 찾지 못한 기사는 오류 파일에만 기록
            append_jsonl(
                errors_path,
                {
                    "title": title,
                    "original_url": article_url,
                    "error": result["error"],
                },
            )
            failure_count += 1

    except Exception as error:
        append_jsonl(
            errors_path,
            {
                "title": title,
                "original_url": article_url,
                "error": f"{type(error).__name__}: {error}",
            },
        )
        failure_count += 1

    if number % 20 == 0:
        print(
            f"{number}/{len(remaining)}건 처리 · "
            f"저장 {success_count}건 · 제외 {failure_count}건"
        )

    # 언론사에 연속 요청을 보내지 않도록 대기
    time.sleep(1)

print("\n이번 작업 완료")
print("이번에 저장한 기사:", success_count, "건")
print("이번에 제외한 기사:", failure_count, "건")
print("중간 저장 파일:", checkpoint_path)

전체 후보: 1787 건
이전에 저장한 기사: 0 건
이전에 제외한 기사: 0 건
이번에 처리할 기사: 1787 건
20/1787건 처리 · 저장 15건 · 제외 5건
40/1787건 처리 · 저장 30건 · 제외 10건
60/1787건 처리 · 저장 46건 · 제외 14건
80/1787건 처리 · 저장 59건 · 제외 21건
100/1787건 처리 · 저장 75건 · 제외 25건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


120/1787건 처리 · 저장 90건 · 제외 30건
140/1787건 처리 · 저장 104건 · 제외 36건
160/1787건 처리 · 저장 120건 · 제외 40건
180/1787건 처리 · 저장 134건 · 제외 46건
200/1787건 처리 · 저장 144건 · 제외 56건
220/1787건 처리 · 저장 158건 · 제외 62건
240/1787건 처리 · 저장 167건 · 제외 73건
260/1787건 처리 · 저장 176건 · 제외 84건
280/1787건 처리 · 저장 189건 · 제외 91건
300/1787건 처리 · 저장 205건 · 제외 95건
320/1787건 처리 · 저장 219건 · 제외 101건
340/1787건 처리 · 저장 226건 · 제외 114건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


360/1787건 처리 · 저장 238건 · 제외 122건
380/1787건 처리 · 저장 252건 · 제외 128건
400/1787건 처리 · 저장 267건 · 제외 133건
420/1787건 처리 · 저장 281건 · 제외 139건
440/1787건 처리 · 저장 295건 · 제외 145건
460/1787건 처리 · 저장 308건 · 제외 152건
480/1787건 처리 · 저장 319건 · 제외 161건
500/1787건 처리 · 저장 336건 · 제외 164건
520/1787건 처리 · 저장 351건 · 제외 169건
540/1787건 처리 · 저장 367건 · 제외 173건
560/1787건 처리 · 저장 380건 · 제외 180건
580/1787건 처리 · 저장 394건 · 제외 186건
600/1787건 처리 · 저장 405건 · 제외 195건
620/1787건 처리 · 저장 418건 · 제외 202건
640/1787건 처리 · 저장 432건 · 제외 208건
660/1787건 처리 · 저장 447건 · 제외 213건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


... (11 outputs left out)

Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


1740/1787건 처리 · 저장 1186건 · 제외 554건


Some characters could not be decoded, and were replaced with REPLACEMENT CHARACTER.


1760/1787건 처리 · 저장 1202건 · 제외 558건
1780/1787건 처리 · 저장 1217건 · 제외 563건

이번 작업 완료
이번에 저장한 기사: 1223 건
이번에 제외한 기사: 564 건
중간 저장 파일: /Users/anna/bulsae/private_data/innisfree_news_checkpoint.jsonl
